# Phase 5 — Legal Statute Identification (LSI): Experiment 2 = Phase 1 **GQA decoder** + **LoRMA** (scratch implementation)

**What this notebook does.** Takes the Phase 1 *decoder-only* GQA language model (`best_model.pt`, 139 M parameters, causal attention, RoPE, SwiGLU, tied embeddings), **freezes it completely**, and adapts it to multi-label statute identification with **LoRMA₊ — low-rank *multiplicative* adaptation, implemented from scratch** (`W' = (I + (α/r)·B·A)·W₀`, Bihany et al. 2025, §3.2.2, additive rank inflation), exactly the scratch implementation used in Assignment 4 (4.4), adapted from GPT-2 `Conv1D` to the `nn.Linear` layers of the GQA decoder.

**What is *not* repeated.** The pooling / head / loss / long-document experiments were done in Experiment 1 (encoder + full fine-tuning) and their winners are fixed here as the recipe:

| Decision (from Exp. 1, not re-tested) | Value |
|---|---|
| Long-document representation | two 510-token windows per case: `head` + `cue_rest` (cue-selected sentences from the part of the case beyond the head window), pooled jointly over all tokens |
| Pooling / head | single-query **attention pooling** → dropout → linear (100 logits, bias initialised to the label prior). *Decoder-specific addition tested once (probe 2):* append a closing `</s>` to every window and concatenate its hidden state — the only position of a causal model that has seen the whole window — to the attention-pooled vector; the better of the two is kept |
| Loss / class weighting | BCE with per-statute positive weights `sqrt(neg/pos)` clipped to [1, 20] |
| Thresholds / post-processing | chosen out-of-fold on dev by the same selection routine (cheap, no training) |

**What *is* new and therefore verified here:** (1) the decoder port is checked against the Phase 1 language-model loss, (2) the LoRMA implementation is checked in float64 (dense vs re-ordered output **and gradients**, base-preserving initialisation, gradient pattern, SVD ranks, merge exactness), (3) two *compact* probes on a 2-epoch subset: probe 1 = LoRMA-specific choices (which matrices to adapt, learning rate — these did not exist in Exp. 1); probe 2 = decoder-specific pooling (attention pooling alone vs attention pooling + last-token state), (4) the final run: **6 epochs, early stopping with patience 2 on dev**, saves the `.pt` files and `submission.json`.

## 0. Configuration (edit this cell only)

In [1]:
import os, sys, re, gc, json, math, time, heapq, random, subprocess, warnings, copy
from pathlib import Path
from dataclasses import dataclass, fields, asdict
from collections import Counter, OrderedDict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------- environment -----------------------------
ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./lsi_work")
WORK.mkdir(parents=True, exist_ok=True)

# Phase 1 GQA decoder artifacts ("" = auto-discover best_model.pt under /kaggle/input or WORK)
PHASE1_GQA_CHECKPOINT_OVERRIDE = "/kaggle/input/datasets/vishwasp23/phase1-gqa/best_model.pt"            # e.g. "/kaggle/input/phase1-gqa/best_model.pt"
PHASE1_TOKENIZER_OVERRIDE = "/kaggle/input/datasets/vishwasp23/tokenizer-p1/legal_bpe_a1_50k_final.json"
# Optional: folder holding the token cache built by Experiment 1 (train_tokens.npy, ...). Saves ~15 min of tokenisation.
EXTERNAL_TOKEN_CACHE = ""

# Data: either a local folder with train.json / dev.json / test.json, or download from Hugging Face.
LOCAL_DATA_DIR = ""
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
HF_FILES = {"train": "Phase_5_LSI/Train/train.json",
            "dev":   "Phase_5_LSI/Train/dev.json",
            "test":  "Phase_5_LSI/Test/test.json"}
# The HF token is read from env HF_TOKEN, then the Kaggle secret "HF_TOKEN", then a prompt. Never hard-code it.

NUM_LABELS = 100
MAX_LEN = 512                  # decoder context (the Phase 1 model was trained with 512)
CONTENT = MAX_LEN - 1          # 1 prefix token + 511 content tokens per window
MAX_DOC_TOKENS = 16_384
TOK_CACHE_VERSION = "v1"       # must match Exp. 1 to reuse its cache

# ----------------------------- recipe fixed by Experiment 1 (NOT re-tested) -----------------------------
EXP1_RECIPE = dict(views=("head", "cue_rest"), view_agg="tokens", head="attn", loss="bce_posw",
                   head_lr=1e-3, dropout=0.1, warmup=0.06, wd=0.01, end_token=False)

# ----------------------------- LoRMA settings -----------------------------
ALL_TARGETS = ("q", "k", "v", "o", "gate", "up", "down")      # q,k,v,o = attention projections; gate,up,down = SwiGLU MLP
LORMA_DEFAULT = dict(rank=8, alpha=16, targets=ALL_TARGETS, lorma_lr=5e-4, a_std=None)   # a_std=None -> 1/sqrt(d_out)
RUN_LORMA_PROBE = True         # compact LoRMA-specific probe (targets x lr); set False to use LORMA_DEFAULT directly
PROBE_GRID = OrderedDict([     # label -> overrides ; every probe = 2 epochs on the experiment subset, same recipe otherwise
    ("all7 | lr 3e-4",   dict(targets=ALL_TARGETS, lorma_lr=3e-4)),
    ("all7 | lr 1e-3",   dict(targets=ALL_TARGETS, lorma_lr=1e-3)),
    ("attn4 | lr 1e-3",  dict(targets=("q", "k", "v", "o"), lorma_lr=1e-3)),
])

# Decoder-specific pooling probe: attention pooling alone (Exp. 1 head)  vs  attention pooling + hidden state of a closing </s> token
RUN_POOL_PROBE = True
POOL_PROBE_GRID = OrderedDict([
    ("pool=attn (Exp.1 head)",                      dict(head="attn",      end_token=False)),
    ("pool=attn + last-token (closing </s>)",       dict(head="attn_last", end_token=True)),
])

# ----------------------------- experiment (probe) budget -----------------------------
EXP_EPOCHS      = 2
EXP_TRAIN_DOCS  = 8_000
EXP_DEV_DOCS    = 3_000
MIN_POS_PER_LABEL_SUBSET = 60
EFFECTIVE_BATCH = 32           # documents per optimizer step
SELECTION_TOL   = 0.004        # parsimony rule: among probes within TOL of the best macro-F1 take the cheapest

# ----------------------------- final training (as requested: 6 epochs, early stopping patience 2) -----------------------------
FINAL_MAX_EPOCHS   = 6
FINAL_PATIENCE     = 2
RESUME_FINAL       = False     # True -> continue an interrupted final run from final_last.pt
SAVE_MERGED_FULL   = True      # also save a self-contained .pt with the LoRMA update merged into the backbone weights
SMOKE_TEST = False             # True -> tiny end-to-end dry run to verify the whole pipeline
FINAL_TRAIN_DOCS_LIMIT = None  # (normally None = all training docs)

SEED = 42
BASE = dict(**EXP1_RECIPE, **LORMA_DEFAULT)

if SMOKE_TEST:
    EXP_TRAIN_DOCS, EXP_DEV_DOCS, FINAL_MAX_EPOCHS, FINAL_TRAIN_DOCS_LIMIT, MIN_POS_PER_LABEL_SUBSET = 256, 256, 2, 512, 2
    PROBE_GRID = OrderedDict(list(PROBE_GRID.items())[:2])
RUN_DIR = WORK / ("gqa_lorma_runs_smoke" if SMOKE_TEST else "gqa_lorma_runs"); RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = WORK / f"lsi_tok_cache_{TOK_CACHE_VERSION}"
EXP_STORE = RUN_DIR / "experiments.json"
print("WORK:", WORK.resolve()); print("ON_KAGGLE:", ON_KAGGLE)

WORK: /kaggle/working
ON_KAGGLE: True


## 1. Imports, GPU, seeds

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    display = print

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(SEED)

assert torch.cuda.is_available(), "Enable a GPU runtime."
device = torch.device("cuda")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name, f"| {gpu.total_memory/1024**3:.1f} GiB | torch {torch.__version__}")

GPU: Tesla T4 | 14.6 GiB | torch 2.11.0+cu128


## 2. Locate / download the Phase 5 data, the Phase 1 GQA checkpoint, the tokenizer and (optionally) the Exp. 1 token cache

In [3]:
try:
    import ijson
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ijson"]); import ijson
    except Exception:
        ijson = None

def _get_hf_token():
    tok = os.environ.get("HF_TOKEN", "").strip()
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = ""
    if not tok:
        from getpass import getpass
        tok = getpass("Hugging Face token: ")
    return tok

def locate_data():
    paths = {}
    if LOCAL_DATA_DIR:
        for split in HF_FILES:
            p = Path(LOCAL_DATA_DIR) / f"{split}.json"
            if not p.is_file(): raise FileNotFoundError(p)
            paths[split] = p
        return paths
    from huggingface_hub import hf_hub_download
    token = _get_hf_token()
    for split, fn in HF_FILES.items():
        paths[split] = Path(hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=fn, token=token))
    return paths

def iter_json_records(path):
    """Yield records from a JSON array (streamed with ijson when available) or JSONL."""
    with open(path, "rb") as f:
        first = b""
        while True:
            ch = f.read(1)
            if not ch: break
            if ch not in b" \t\r\n": first = ch; break
        f.seek(0)
        if first == b"[":
            if ijson is not None: yield from ijson.items(f, "item")
            else: yield from json.load(f)
        elif first == b"{":
            for line in f:
                line = line.strip()
                if line: yield json.loads(line)
        else:
            raise ValueError(f"Unrecognised JSON layout: {path}")

def find_artifact(filename, override=""):
    if override and Path(override).is_file(): return Path(override).resolve()
    hits = set()
    for root in [Path("/kaggle/input"), WORK, Path(".")]:
        if root.exists(): hits.update(p.resolve() for p in root.rglob(filename) if p.is_file())
    hits = sorted(hits)
    if len(hits) != 1:
        raise FileNotFoundError(f"{filename}: found {len(hits)} candidates {hits}. Set the override path in the config cell.")
    return hits[0]

REQUIRED_CACHE_FILES = [f"{s}_{k}" for s in ("train", "dev", "test") for k in ("tokens.npy", "flags.npy", "offsets.npy", "orig_len.npy", "case_ids.json")] \
                     + ["train_labels.npy", "dev_labels.npy"]
def cache_complete(d): return d is not None and Path(d).is_dir() and all((Path(d) / f).exists() for f in REQUIRED_CACHE_FILES)

# token cache: own working copy -> user-provided folder -> any Kaggle dataset containing train_tokens.npy
if not cache_complete(CACHE_DIR):
    cands = [Path(EXTERNAL_TOKEN_CACHE)] if EXTERNAL_TOKEN_CACHE else []
    if Path("/kaggle/input").exists(): cands += [p.parent for p in Path("/kaggle/input").rglob("train_tokens.npy")]
    for c in cands:
        if cache_complete(c): CACHE_DIR = c; print("re-using token cache from Experiment 1:", c); break
CACHE_DIR.mkdir(parents=True, exist_ok=True) if not cache_complete(CACHE_DIR) else None

need_raw = not cache_complete(CACHE_DIR)
paths = locate_data() if need_raw else {}
PHASE1_CHECKPOINT_PATH = find_artifact("best_model.pt", PHASE1_GQA_CHECKPOINT_OVERRIDE)
PHASE1_TOKENIZER_PATH  = find_artifact("legal_bpe_a1_50k_final.json", PHASE1_TOKENIZER_OVERRIDE)
print("token cache:", CACHE_DIR, "| complete:", cache_complete(CACHE_DIR))
print("data paths :", {k: str(v) for k, v in paths.items()} if paths else "(token cache present, raw files not needed)")
print("GQA checkpoint:", PHASE1_CHECKPOINT_PATH); print("tokenizer     :", PHASE1_TOKENIZER_PATH)

Hugging Face token:  ········


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

token cache: /kaggle/working/lsi_tok_cache_v1 | complete: False
data paths : {'train': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/train.json', 'dev': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/dev.json', 'test': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Test/test.json'}
GQA checkpoint: /kaggle/input/datasets/vishwasp23/phase1-gqa/best_model.pt
tokenizer     : /kaggle/input/datasets/vishwasp23/tokenizer-p1/legal_bpe_a1_50k_final.json


## 3. Phase 1 tokenizer (A-1 byte-level BPE, identical to Phase 1 and to Experiment 1)

In [4]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
N_SPECIAL, N_BYTES = 4, 256
BASE_VOCAB = N_SPECIAL + N_BYTES
WORD_RE = re.compile(r"\S+")

class A1BPETokenizer:
    def __init__(self, itos, merges):
        self.itos = list(itos); self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos)
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}

    def _encode_word(self, word):
        raw = word.encode("utf-8")
        seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks: return seq
        n = len(seq); prev = [-1] + list(range(n - 1)); nxt = list(range(1, n)) + [-1]
        active = [True] * n; heap = []
        for i in range(n - 1):
            r = self._merge_ranks.get((seq[i], seq[i + 1]))
            if r is not None: heapq.heappush(heap, (r, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]: continue
            if nxt[left] != right or prev[right] != left: continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank: continue
            seq[left] = BASE_VOCAB + rank; active[right] = False
            rn = nxt[right]; nxt[left] = rn
            if rn != -1: prev[rn] = left
            lp = prev[left]
            if lp != -1:
                nr = self._merge_ranks.get((seq[lp], seq[left]))
                if nr is not None: heapq.heappush(heap, (nr, lp, left))
            if rn != -1:
                nr = self._merge_ranks.get((seq[left], seq[rn]))
                if nr is not None: heapq.heappush(heap, (nr, left, rn))
        out, i = [], 0
        while i != -1:
            if active[i]: out.append(seq[i])
            i = nxt[i]
        return out

def load_a1_tokenizer(path):
    with open(path, "r", encoding="utf-8") as f: payload = json.load(f)
    tok = A1BPETokenizer(payload["itos"], payload["merges"])
    assert tok.vocab_size == 50_000 and tok.itos[:4] == SPECIAL_TOKENS
    return tok

phase1_tokenizer = load_a1_tokenizer(PHASE1_TOKENIZER_PATH)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPACE_ID = N_SPECIAL + 32
# The Phase 1 decoder was pre-trained on a token STREAM in which every document is followed by </s> (id 3) and <s> (id 2)
# never occurs. We therefore start every window with </s> (the "document boundary" token the model actually saw before a
# document) instead of the never-trained <s>.
PREFIX_ID = EOS_ID
print("tokenizer OK | prefix token id:", PREFIX_ID)

tokenizer OK | prefix token id: 3


## 4. Cache the *full* tokenised documents once (re-used from Experiment 1 when available)

In [5]:
CUE_RE = re.compile(r"<(?:SECTION|ACT)>|^(?:[Ss]ections?|[Ss]ec\.?|[Aa]rticles?|[Rr]ules?|[Aa]ct|[Aa]cts|IPC|CrPC|Cr\.P\.C\.?|Code)\W*$")
SENT_END_CHARS = (".", ";", "?", "!", ":")
FLAG_CUE, FLAG_SENT = 1, 2

class FastDocTokenizer:
    """Word-level cached version of the Phase 1 tokenizer. Output ids are identical to the original encoder
    except runs of whitespace between words are collapsed to ONE space token and leading/trailing whitespace is dropped."""
    def __init__(self, tok, max_doc_tokens=MAX_DOC_TOKENS, tail_keep=4096, cache_limit=3_000_000):
        self.tok, self.max_doc_tokens, self.tail_keep, self.cache_limit = tok, max_doc_tokens, tail_keep, cache_limit
        self.cache = {}
    def _word(self, w):
        r = self.cache.get(w)
        if r is None:
            ids = self.tok._encode_word(w)
            fl = [0] * len(ids)
            if CUE_RE.search(w): fl[0] |= FLAG_CUE
            if w.endswith(SENT_END_CHARS): fl[-1] |= FLAG_SENT
            r = (tuple(ids), tuple(fl))
            if len(self.cache) > self.cache_limit: self.cache.clear()
            self.cache[w] = r
        return r
    def encode(self, text):
        ids, fl, cursor, first = [], [], 0, True
        for m in WORD_RE.finditer(text):
            if not first: ids.append(SPACE_ID); fl.append(0)
            w_ids, w_fl = self._word(m.group(0)); ids.extend(w_ids); fl.extend(w_fl); first = False
        n = len(ids)
        if n > self.max_doc_tokens:
            cut = self.max_doc_tokens - self.tail_keep
            ids = ids[:cut] + ids[-self.tail_keep:]; fl = fl[:cut] + fl[-self.tail_keep:]
        return np.asarray(ids, dtype=np.uint16), np.asarray(fl, dtype=np.uint8), n

_WORKER = {}
def _init_worker():
    _WORKER["t"] = FastDocTokenizer(phase1_tokenizer)
def _tok_batch(texts):
    t = _WORKER["t"]; out = []
    for s in texts:
        i, f, n = t.encode(s); out.append((i, f, n))
    return out

def build_split_cache(split):
    flag = CACHE_DIR / f"{split}_done.flag"
    if all((CACHE_DIR / f"{split}_{k}").exists() for k in ("tokens.npy", "flags.npy", "offsets.npy", "orig_len.npy", "case_ids.json")): return
    has_labels = split != "test"
    ids_raw, texts, labels = [], [], []
    for row in tqdm(iter_json_records(paths[split]), desc=f"read {split}"):
        ids_raw.append(row["id"]); texts.append(row["text"])
        if has_labels:
            lab = sorted(set(int(x) for x in row["labels"]))
            assert all(0 <= x < NUM_LABELS for x in lab), f"bad label in {row['id']}"
            labels.append(lab)
    n_docs = len(texts)
    batches = [texts[i:i + 128] for i in range(0, n_docs, 128)]
    results = []
    try:
        import multiprocessing as mp
        ctx = mp.get_context("fork"); nproc = min(4, os.cpu_count() or 1)
        with ctx.Pool(nproc, initializer=_init_worker) as pool:
            for r in tqdm(pool.imap(_tok_batch, batches), total=len(batches), desc=f"tokenise {split}"):
                results.extend(r)
    except Exception as e:
        print("parallel tokenisation unavailable (", type(e).__name__, ") -> sequential"); _init_worker()
        results = []
        for b in tqdm(batches, desc=f"tokenise {split} (seq)"): results.extend(_tok_batch(b))
    lengths = np.array([len(r[0]) for r in results], dtype=np.int64)
    offsets = np.zeros(n_docs + 1, dtype=np.int64); offsets[1:] = np.cumsum(lengths)
    np.save(CACHE_DIR / f"{split}_tokens.npy", np.concatenate([r[0] for r in results]))
    np.save(CACHE_DIR / f"{split}_flags.npy",  np.concatenate([r[1] for r in results]))
    np.save(CACHE_DIR / f"{split}_offsets.npy", offsets)
    np.save(CACHE_DIR / f"{split}_orig_len.npy", np.array([r[2] for r in results], dtype=np.int64))
    if has_labels:
        Y = np.zeros((n_docs, NUM_LABELS), dtype=np.uint8)
        for i, lab in enumerate(labels): Y[i, lab] = 1
        np.save(CACHE_DIR / f"{split}_labels.npy", Y)
    with open(CACHE_DIR / f"{split}_case_ids.json", "w", encoding="utf-8") as f: json.dump(ids_raw, f)
    flag.write_text("ok"); del results, texts; gc.collect()
    print(f"{split}: {n_docs:,} docs | {lengths.sum():,} tokens | mean {lengths.mean():.0f} | median {np.median(lengths):.0f}")

t0 = time.time()
for s in ["train", "dev", "test"]: build_split_cache(s)
print(f"token cache ready in {(time.time()-t0)/60:.1f} min  ->  {CACHE_DIR}")

read train: 0it [00:00, ?it/s]

tokenise train:   0%|          | 0/334 [00:00<?, ?it/s]

train: 42,750 docs | 102,823,711 tokens | mean 2405 | median 988


read dev: 0it [00:00, ?it/s]

tokenise dev:   0%|          | 0/80 [00:00<?, ?it/s]

dev: 10,181 docs | 27,323,268 tokens | mean 2684 | median 1247


read test: 0it [00:00, ?it/s]

tokenise test:   0%|          | 0/102 [00:00<?, ?it/s]

test: 13,019 docs | 33,541,917 tokens | mean 2576 | median 1227
token cache ready in 1.0 min  ->  /kaggle/working/lsi_tok_cache_v1


### 4b. Quick data diagnostics

In [6]:
def load_arrays(split, has_labels=True):
    d = dict(tokens=np.load(CACHE_DIR / f"{split}_tokens.npy", mmap_mode="r"),
             flags=np.load(CACHE_DIR / f"{split}_flags.npy", mmap_mode="r"),
             offsets=np.load(CACHE_DIR / f"{split}_offsets.npy"),
             orig_len=np.load(CACHE_DIR / f"{split}_orig_len.npy"))
    d["ids"] = json.load(open(CACHE_DIR / f"{split}_case_ids.json"))
    if has_labels: d["labels"] = np.load(CACHE_DIR / f"{split}_labels.npy")
    return d

ARR = {"train": load_arrays("train"), "dev": load_arrays("dev"), "test": load_arrays("test", has_labels=False)}
for s, a in ARR.items():
    L = np.diff(a["offsets"])
    cue = np.array([int((np.asarray(a["flags"][a["offsets"][i]:a["offsets"][i+1]]) & FLAG_CUE).sum()) for i in range(0, len(L), max(1, len(L)//2000))])
    print(f"{s:5s} docs={len(L):6,d} | tokens p50={np.percentile(L,50):.0f} p90={np.percentile(L,90):.0f} p99={np.percentile(L,99):.0f}"
          f" | >510 tok: {(L>CONTENT).mean()*100:.1f}% | >1020 tok: {(L>2*CONTENT).mean()*100:.1f}% | cue words/doc (sample) mean={cue.mean():.1f}")
Ytr, Ydv = ARR["train"]["labels"], ARR["dev"]["labels"]
print("labels/doc train: mean %.2f | positives per label min/median/max = %d/%d/%d" % (Ytr.sum(1).mean(), Ytr.sum(0).min(), np.median(Ytr.sum(0)), Ytr.sum(0).max()))
PRIOR = np.clip(Ytr.mean(0), 1e-5, 1 - 1e-5)                 # label prior (train) used for bias init and class weights
POS_COUNT = Ytr.sum(0).astype(np.float64)

train docs=42,750 | tokens p50=988 p90=6383 p99=16384 | >510 tok: 70.9% | >1020 tok: 49.0% | cue words/doc (sample) mean=11.5
dev   docs=10,181 | tokens p50=1247 p90=6885 p99=16384 | >510 tok: 73.1% | >1020 tok: 55.2% | cue words/doc (sample) mean=12.5
test  docs=13,019 | tokens p50=1227 p90=6538 p99=16384 | >510 tok: 73.6% | >1020 tok: 54.7% | cue words/doc (sample) mean=12.1
labels/doc train: mean 3.70 | positives per label min/median/max = 272/679/11730


## 5. The Phase 1 GQA decoder (causal, RoPE, SwiGLU, tied embeddings) — port + verification

The classes below re-implement the Phase 1 `LegalDecoderLM` forward pass with identical parameter names (`token_embedding`, `blocks.i.{norm1, attn.{q,k,v,out}_proj, norm2, mlp.{gate,up,down}_proj}`, `final_norm`) so the Phase 1 weights load with `strict=True`. Two cosmetic differences: attention uses PyTorch's fused causal SDPA instead of the hand-written masked softmax (mathematically identical, tested below), and no KV-cache path (not needed for classification).

Because attention is **causal**, right-padding can never influence real tokens, so no padding mask is required; pooling masks exclude padding. A window starts with `</s>` (id 3) because that is the document-boundary token the model saw before every document during pre-training.

In [7]:
_ckpt = torch.load(PHASE1_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
GQA_STATE = {k: v for k, v in _ckpt["model_state_dict"].items()}
_cfg_dict = dict(_ckpt["model_config"])
print("Phase 1 checkpoint: global step", _ckpt.get("global_step"), "| tokens seen", f"{_ckpt.get('tokens_seen', 0):,}", "| best val loss", _ckpt.get("best_val_loss"))
del _ckpt; gc.collect()

@dataclass
class ModelConfig:
    vocab_size: int = 50_000; d_model: int = 768; n_layers: int = 16; n_heads: int = 12
    n_kv_heads: int = 4; d_ff: int = 2048; max_seq_len: int = 512
    rope_base: float = 10_000.0; rms_norm_eps: float = 1e-6; dropout: float = 0.0

cfg = ModelConfig(**{k: v for k, v in _cfg_dict.items() if k in {f.name for f in fields(ModelConfig)}})
print(cfg)
assert cfg.max_seq_len >= MAX_LEN

def build_rope_cache(max_seq_len, d_head, base=10_000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    ang = torch.cat([freqs, freqs], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos.to(x.device, x.dtype)[None, None] + rotate_half(x) * sin.to(x.device, x.dtype)[None, None]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        dt = x.dtype; xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dt)

class SwiGLU(nn.Module):
    def __init__(self, d, ff):
        super().__init__()
        self.gate_proj = nn.Linear(d, ff, bias=False); self.up_proj = nn.Linear(d, ff, bias=False)
        self.down_proj = nn.Linear(ff, d, bias=False)
    def forward(self, x): return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class CausalGQAAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.d_model % c.n_heads == 0 and c.n_heads % c.n_kv_heads == 0
        self.d_model, self.n_heads, self.n_kv_heads = c.d_model, c.n_heads, c.n_kv_heads
        self.head_dim = c.d_model // c.n_heads; self.n_rep = c.n_heads // c.n_kv_heads; self.max_seq_len = c.max_seq_len
        self.q_proj = nn.Linear(c.d_model, c.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(c.d_model, c.d_model, bias=False)
        cos, sin = build_rope_cache(c.max_seq_len, self.head_dim, c.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
    def _split(self, x, n):
        b, t, _ = x.shape
        return x.reshape(b, t, n, self.head_dim).transpose(1, 2)
    def forward(self, x):
        b, t, _ = x.shape
        if t > self.max_seq_len: raise ValueError(f"sequence {t} > {self.max_seq_len}")
        q = apply_rope(self._split(self.q_proj(x), self.n_heads), self.rope_cos[:t], self.rope_sin[:t])
        k = apply_rope(self._split(self.k_proj(x), self.n_kv_heads), self.rope_cos[:t], self.rope_sin[:t])
        v = self._split(self.v_proj(x), self.n_kv_heads)
        k = k.repeat_interleave(self.n_rep, dim=1); v = v.repeat_interleave(self.n_rep, dim=1)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
        return self.out_proj(a.transpose(1, 2).contiguous().view(b, t, self.d_model))

class DecoderBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm1 = RMSNorm(c.d_model, c.rms_norm_eps); self.attn = CausalGQAAttention(c)
        self.norm2 = RMSNorm(c.d_model, c.rms_norm_eps); self.mlp = SwiGLU(c.d_model, c.d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class LegalDecoder(nn.Module):
    """Phase 1 backbone: returns the final-norm hidden states [B, T, d_model] (the tied LM head is `lm_logits`)."""
    def __init__(self, c, grad_ckpt=False):
        super().__init__(); self.config = c; self.grad_ckpt = grad_ckpt
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([DecoderBlock(c) for _ in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)
    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for blk in self.blocks:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(blk, x, use_reentrant=False)
            else:
                x = blk(x)
        return self.final_norm(x)
    def lm_logits(self, hidden): return F.linear(hidden, self.token_embedding.weight)      # tied head

def load_gqa_weights(model, state):
    cleaned = {}
    for k, v in state.items():
        for pre in ("module.", "_orig_mod."):
            if k.startswith(pre): k = k[len(pre):]
        cleaned[k] = v
    cleaned.pop("lm_head.weight", None)                     # tied to token_embedding.weight
    model.load_state_dict(cleaned, strict=True)
    return model

_probe = LegalDecoder(cfg); load_gqa_weights(_probe, GQA_STATE)
BACKBONE_PARAMS = sum(p.numel() for p in _probe.parameters()); del _probe
print(f"Phase 1 GQA decoder loaded with strict=True: {BACKBONE_PARAMS:,} parameters (expected 139,088,640)")
assert BACKBONE_PARAMS == 139_088_640

Phase 1 checkpoint: global step 15000 | tokens seen 491,651,072 | best val loss 1.4796594381332397
ModelConfig(vocab_size=50000, d_model=768, n_layers=16, n_heads=12, n_kv_heads=4, d_ff=2048, max_seq_len=512, rope_base=10000.0, rms_norm_eps=1e-06, dropout=0.0)
Phase 1 GQA decoder loaded with strict=True: 139,088,640 parameters (expected 139,088,640)


### 5b. Verification of the decoder port (tiny random model on CPU + the real checkpoint's language-model loss)

In [8]:
def ref_causal_attention(attn, x):
    """Reference: the hand-written masked-softmax attention of the Phase 1 notebook (no KV cache)."""
    b, t, _ = x.shape
    q = apply_rope(attn._split(attn.q_proj(x), attn.n_heads), attn.rope_cos[:t], attn.rope_sin[:t])
    k = apply_rope(attn._split(attn.k_proj(x), attn.n_kv_heads), attn.rope_cos[:t], attn.rope_sin[:t])
    v = attn._split(attn.v_proj(x), attn.n_kv_heads)
    k = k.repeat_interleave(attn.n_rep, dim=1); v = v.repeat_interleave(attn.n_rep, dim=1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(attn.head_dim)
    mask = torch.ones(t, t, dtype=torch.bool, device=x.device).triu(diagonal=1)
    w = F.softmax(scores.masked_fill(mask, torch.finfo(scores.dtype).min), dim=-1)
    return attn.out_proj(torch.matmul(w, v).transpose(1, 2).contiguous().reshape(b, t, attn.d_model))

def test_decoder_port():
    torch.manual_seed(0)
    tcfg = ModelConfig(vocab_size=120, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, d_ff=96, max_seq_len=32)
    dec = LegalDecoder(tcfg)
    for p in dec.parameters(): nn.init.normal_(p, std=0.2)
    x = torch.randn(3, 20, 64)
    attn = dec.blocks[0].attn
    err = (attn(x) - ref_causal_attention(attn, x)).abs().max().item(); assert err < 1e-5, err
    ids = torch.randint(4, 120, (3, 20)); h = dec(ids)
    ids2 = ids.clone(); ids2[:, 12:] = torch.randint(4, 120, (3, 8)); h2 = dec(ids2)          # change the future
    assert (h[:, :12] - h2[:, :12]).abs().max().item() < 1e-5, "attention is not causal"
    padded = torch.cat([ids, torch.zeros(3, 8, dtype=torch.long)], 1); hp = dec(padded)       # right padding
    assert (h - hp[:, :20]).abs().max().item() < 1e-5, "right padding leaks into real tokens"
    print(f"decoder port OK: SDPA-vs-reference max err {err:.1e} | causal | right-padding invariant")
test_decoder_port()

@torch.no_grad()
def lm_sanity_check(n_docs=48):
    """The real Phase 1 checkpoint must reproduce a low next-token loss on real legal text (random weights give ~10.8)."""
    dec = LegalDecoder(cfg).to(device); load_gqa_weights(dec, GQA_STATE); dec.eval()
    a = ARR["train"]; rng = np.random.default_rng(1); losses, shuffled = [], []
    cands = [d for d in rng.permutation(len(a["offsets"]) - 1)[:2000] if a["offsets"][d + 1] - a["offsets"][d] >= 520][:n_docs]
    for d in cands:
        tok = np.asarray(a["tokens"][a["offsets"][d]:a["offsets"][d] + 300 + 211]).astype(np.int64)
        ids = torch.from_numpy(np.concatenate([[PREFIX_ID], tok])).long()[None].to(device)           # [</s>, 511 tokens]
        with torch.autocast("cuda", dtype=torch.float16):
            logits = dec.lm_logits(dec(ids))
        losses.append(F.cross_entropy(logits[0, 15:-1].float(), ids[0, 16:]).item())
        perm = ids.clone(); perm[0, 1:] = ids[0, 1:][torch.randperm(ids.shape[1] - 1, device=device)]
        with torch.autocast("cuda", dtype=torch.float16):
            lg2 = dec.lm_logits(dec(perm))
        shuffled.append(F.cross_entropy(lg2[0, 15:-1].float(), perm[0, 16:]).item())
    del dec; torch.cuda.empty_cache()
    return float(np.mean(losses)), float(np.mean(shuffled))

_l, _ls = lm_sanity_check()
print(f"next-token loss of the loaded Phase 1 decoder on real Phase 5 case text: {_l:.3f}  (token-shuffled control {_ls:.3f}; random weights ~10.8)")
assert _l < 5.0 and _ls > _l + 1.0, "decoder port does not reproduce the Phase 1 language model -> check the checkpoint / architecture"
print("decoder port reproduces the pre-trained language model")

decoder port OK: SDPA-vs-reference max err 1.4e-06 | causal | right-padding invariant
next-token loss of the loaded Phase 1 decoder on real Phase 5 case text: 2.003  (token-shuffled control 7.810; random weights ~10.8)
decoder port reproduces the pre-trained language model


## 6. LoRMA₊ from scratch (adapted from the Assignment 4 implementation)

**Method.** For a frozen linear map `W₀ ∈ ℝ^{d_out×d_in}` LoRMA replaces LoRA's additive update by a *multiplicative* one: `W' = (I + (α/r)·B·A)·W₀`, `A ∈ ℝ^{r×d_out}`, `B ∈ ℝ^{d_out×r}`. The identity term is the *additive rank inflation* (LoRMA₊): a naive `(BA)W₀` has rank ≤ r and cannot even represent "no change"; `I + (α/r)BA` is full rank. Initialisation `B = 0` (A random) gives `W' = W₀` exactly. The forward pass is **re-ordered** so the `d_out × d_out` matrix is never built: `y = W₀x + (α/r)·B(A(W₀x))`.

**Changes w.r.t. the Assignment-4 code (all other behaviour and signatures kept):**
1. Wraps `nn.Linear` (`y = xW₀ᵀ`, GQA layers have no bias) instead of GPT-2 `Conv1D` (stored `[in, out]`). If a bias exists it is added *after* the transformation and is not multiplied.
2. Separate q / k / v / o / gate / up / down wrappers instead of one fused `c_attn`; each has its own independent `(A, B)` pair.
3. **`A` is initialised with std `1/√d_out`** (assignment: std 1) via the new optional `a_std` argument (`a_std=1.0` reproduces the assignment). With unit-variance `A`, `A·(W₀x)` is `√d_out` times larger than `W₀x` itself, so the very first Adam steps of `B` would rewrite the layer; the numbers are printed in the next cell. The prototype tests below run with *both* settings.
4. `merged_linear()` returns an ordinary `nn.Linear` holding `W' = W₀ + (α/r)·B(A W₀)` — LoRMA is mergeable exactly like LoRA when there is no bias (zero inference overhead after merging).

In [9]:
LORMA_VARIANT = {"name": "LoRMA+", "equation": "W' = (I + (alpha/r) B A) W_0",
                 "paper_version": "Bihany, Patel & Modi (2025), arXiv:2506.07621, Sec. 3.2.2 Additive Rank Inflation", "alpha": 16}

def _lorma_scaling(rank, variant_config=None):
    config = LORMA_VARIANT if variant_config is None else variant_config
    return config.get("alpha", 16) / rank

def make_lorma_factors(d_in, d_out, rank, init=False, generator=None, dtype=torch.float64, swap_ab=False, a_std=1.0, b_std=1.0):
    """Returns (A [r, d_out], B [d_out, r]).  init=False -> both random.  init=True -> base-preserving:
    standard (swap_ab=False): B = 0, A random ;  swapped (swap_ab=True): A = 0, B random."""
    if rank < 1: raise ValueError("rank must be positive")
    a = torch.randn(rank, d_out, generator=generator, dtype=dtype) * a_std
    b = torch.randn(d_out, rank, generator=generator, dtype=dtype) * b_std
    if init:
        if swap_ab: a = torch.zeros_like(a)
        else:       b = torch.zeros_like(b)
    return a, b

def lorma_dense(x, frozen_weight, factors, variant_config=None):
    """Dense form: builds the full d_out x d_out matrix.  frozen_weight: [d_out, d_in]."""
    a, b = factors; scale = _lorma_scaling(a.size(0), variant_config)
    identity = torch.eye(frozen_weight.size(0), dtype=frozen_weight.dtype, device=frozen_weight.device)
    return x @ ((identity + scale * (b @ a)) @ frozen_weight).T

def lorma_reordered(x, frozen_weight, factors, variant_config=None):
    """Re-ordered form: never materialises a d_out x d_out product."""
    a, b = factors; scale = _lorma_scaling(a.size(0), variant_config)
    base_output = x @ frozen_weight.T
    return base_output + scale * (base_output @ a.T @ b.T)

class LoRMALinear(nn.Module):
    def __init__(self, base_linear, rank=8, variant_config=None, swap_ab=False, a_std=None):
        super().__init__()
        if rank < 1: raise ValueError("rank must be positive")
        self.base = base_linear
        for p in self.base.parameters(): p.requires_grad_(False)
        self.rank = rank; self.variant_config = LORMA_VARIANT if variant_config is None else variant_config
        self.swap_ab = swap_ab
        d_in, d_out = self.base.in_features, self.base.out_features
        a, b = make_lorma_factors(d_in, d_out, rank, init=True, dtype=self.base.weight.dtype, swap_ab=swap_ab,
                                  a_std=(1.0 / math.sqrt(d_out)) if a_std is None else a_std, b_std=1.0 / math.sqrt(rank))
        self.lorma_A = nn.Parameter(a.to(self.base.weight.device)); self.lorma_B = nn.Parameter(b.to(self.base.weight.device))
    @property
    def scaling(self): return _lorma_scaling(self.rank, self.variant_config)
    def forward(self, x):
        base_out = F.linear(x, self.base.weight)                       # W0 x   (the bias, if any, is not transformed)
        out = base_out + self.scaling * ((base_out @ self.lorma_A.t()) @ self.lorma_B.t())
        return out if self.base.bias is None else out + self.base.bias
    @torch.no_grad()
    def merged_linear(self):
        W0 = self.base.weight; delta = self.scaling * (self.lorma_B @ (self.lorma_A @ W0))      # (alpha/r) B (A W0)
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(W0.device, W0.dtype)
        lin.weight.copy_(W0 + delta)
        if self.base.bias is not None: lin.bias.copy_(self.base.bias)
        for p in lin.parameters(): p.requires_grad_(False)
        return lin

TARGET_MAP = {"q": ("attn", "q_proj"), "k": ("attn", "k_proj"), "v": ("attn", "v_proj"), "o": ("attn", "out_proj"),
              "gate": ("mlp", "gate_proj"), "up": ("mlp", "up_proj"), "down": ("mlp", "down_proj")}

def target_out_dim(c, t):
    hd = c.d_model // c.n_heads
    return {"q": c.n_heads * hd, "k": c.n_kv_heads * hd, "v": c.n_kv_heads * hd, "o": c.d_model,
            "gate": c.d_ff, "up": c.d_ff, "down": c.d_model}[t]

def lorma_param_count(c, targets, rank):
    """Hand-derived: every adapted matrix has A [r, d_out] and B [d_out, r] -> 2*r*d_out parameters."""
    return c.n_layers * sum(2 * rank * target_out_dim(c, t) for t in targets)

def attach_lorma(decoder, targets, rank=8, alpha=16, swap_ab=False, a_std=None):
    for p in decoder.parameters(): p.requires_grad_(False)                       # whole backbone frozen
    vc = {**LORMA_VARIANT, "alpha": alpha}
    for blk in decoder.blocks:
        for t in targets:
            parent_name, child = TARGET_MAP[t]; parent = getattr(blk, parent_name); base = getattr(parent, child)
            assert not isinstance(base, LoRMALinear), "LoRMA already attached"
            setattr(parent, child, LoRMALinear(base, rank, vc, swap_ab, a_std))
    return decoder

def merge_lorma_(module):
    """In-place: replace every LoRMALinear below `module` with its merged nn.Linear."""
    for name, child in list(module.named_children()):
        if isinstance(child, LoRMALinear): setattr(module, name, child.merged_linear())
        else: merge_lorma_(child)
    return module
print("LoRMA ready:", LORMA_VARIANT["name"], "|", LORMA_VARIANT["equation"])

LoRMA ready: LoRMA+ | W' = (I + (alpha/r) B A) W_0


### 6b. Verification of LoRMA (float64 algebra, SVD ranks, gradient pattern, merge exactness, tiny-decoder model tests)

In [10]:
def _num_rank(m):
    s = torch.linalg.svdvals(m.detach().to(torch.float64)); tol = max(m.shape) * torch.finfo(torch.float64).eps * s.max().item()
    return int((s > tol).sum().item())

def check_lorma_math(d_in=12, d_out=16, rank=3, swap_ab=False, a_std=1.0, alpha=16):
    vc = {**LORMA_VARIANT, "alpha": alpha}; dt = torch.float64; g = torch.Generator().manual_seed(2026)
    x0 = torch.randn(4, d_in, generator=g, dtype=dt); w0 = torch.randn(d_out, d_in, generator=g, dtype=dt)
    a0, b0 = make_lorma_factors(d_in, d_out, rank, init=False, generator=g, dtype=dt, a_std=a_std)
    leaves = {n: t.clone().requires_grad_(True) for n, t in dict(x=x0, W=w0, A=a0, B=b0).items()}
    leaves2 = {n: t.clone().requires_grad_(True) for n, t in dict(x=x0, W=w0, A=a0, B=b0).items()}
    yd = lorma_dense(leaves["x"], leaves["W"], (leaves["A"], leaves["B"]), vc)
    yr = lorma_reordered(leaves2["x"], leaves2["W"], (leaves2["A"], leaves2["B"]), vc)
    probe = torch.randn(yd.shape, generator=g, dtype=dt)
    gd = torch.autograd.grad((yd * probe).sum(), list(leaves.values())); gr = torch.autograd.grad((yr * probe).sum(), list(leaves2.values()))
    out_err = (yd - yr).abs().max().item(); grad_err = {n: (a - b).abs().max().item() for n, a, b in zip(leaves, gd, gr)}
    # base-preserving initialisation + step-0 gradient pattern
    ai, bi = make_lorma_factors(d_in, d_out, rank, init=True, generator=g, dtype=dt, swap_ab=swap_ab, a_std=a_std)
    ai, bi = ai.clone().requires_grad_(True), bi.clone().requires_grad_(True)
    yi = lorma_reordered(x0, w0, (ai, bi), vc); init_err = (yi - x0 @ w0.T).abs().max().item()
    ga, gb = torch.autograd.grad((yi * torch.randn(yi.shape, generator=g, dtype=dt)).sum(), (ai, bi))
    pattern_ok = (torch.count_nonzero(ga) > 0 and torch.count_nonzero(gb) == 0) if swap_ab else (torch.count_nonzero(ga) == 0 and torch.count_nonzero(gb) > 0)
    scale = alpha / rank; naive = b0 @ a0; infl = torch.eye(d_out, dtype=dt) + scale * naive
    ranks = dict(A=_num_rank(a0), B=_num_rank(b0), naive_BA=_num_rank(naive), inflated_I_plus_sBA=_num_rank(infl), update_W_minus_W0=_num_rank((infl - torch.eye(d_out, dtype=dt)) @ w0))
    assert out_err < 1e-10 and max(grad_err.values()) < 1e-9, (out_err, grad_err)
    assert init_err == 0.0 and pattern_ok, (init_err, pattern_ok)
    assert ranks["naive_BA"] <= rank and ranks["inflated_I_plus_sBA"] == d_out and ranks["update_W_minus_W0"] <= rank, ranks
    assert a0.numel() + b0.numel() == 2 * rank * d_out
    return dict(swap_ab=swap_ab, a_std=a_std, out_err=out_err, max_grad_err=max(grad_err.values()), init_err=init_err, grad_pattern_ok=bool(pattern_ok), **{f"rank_{k}": v for k, v in ranks.items()})

rows = [check_lorma_math(swap_ab=s, a_std=a) for s in (False, True) for a in (1.0, 16 ** -0.5)]
display(pd.DataFrame(rows).round(2))
print("float64: dense == re-ordered (outputs AND gradients), identity at init, correct step-0 gradient pattern, rank(BA)<=r, rank(I+sBA)=d_out, rank(W'-W0)<=r  -> PASS")

# why a_std = 1/sqrt(d_out): relative change of a layer output after ONE Adam step on B (|B_ij| = lr), y = W0 x of size d_out
torch.manual_seed(0); d_out, r = 768, 8; y = torch.randn(4096, d_out); rel = {}
for a_std in (1.0, d_out ** -0.5):
    A = torch.randn(r, d_out) * a_std; Bm = torch.sign(torch.randn(d_out, r)) * 1e-3
    rel[f"a_std={a_std:.4f}"] = float(((2.0 * (y @ A.T) @ Bm.T).norm(dim=1) / y.norm(dim=1)).mean())
print("relative output change after one Adam step (lr 1e-3) of B:", {k: f"{v:.2%}" for k, v in rel.items()})

def tiny_model_tests():
    torch.manual_seed(1)
    tcfg = ModelConfig(vocab_size=120, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, d_ff=96, max_seq_len=32)
    base = LegalDecoder(tcfg)
    for p in base.parameters(): nn.init.normal_(p, std=0.2)
    ids = torch.randint(4, 120, (3, 20)); h0 = base(ids).detach(); targets = ALL_TARGETS
    for swap in (False, True):
        m = attach_lorma(copy.deepcopy(base), targets, rank=4, alpha=8, swap_ab=swap)
        assert (m(ids) - h0).abs().max().item() < 1e-6, "LoRMA is not the identity at initialisation"
        counted = sum(p.numel() for p in m.parameters() if p.requires_grad); hand = lorma_param_count(tcfg, targets, 4)
        assert counted == hand, (counted, hand)
        frozen0 = {n: p.detach().clone() for n, p in m.named_parameters() if not p.requires_grad}
        opt = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=1e-2)
        m.zero_grad(); m(ids).pow(2).mean().backward()
        for n, p in m.named_parameters():
            if "lorma_A" in n: assert (torch.count_nonzero(p.grad) == 0) != swap, n
            if "lorma_B" in n: assert (torch.count_nonzero(p.grad) > 0) != swap, n
        opt.step(); m.zero_grad(); m(ids).pow(2).mean().backward()
        assert all(torch.count_nonzero(p.grad) > 0 for n, p in m.named_parameters() if p.requires_grad), "after one step both factors must receive gradient"
        assert all(torch.equal(frozen0[n], p.detach()) for n, p in m.named_parameters() if n in frozen0), "frozen weights changed"
    # merged == unmerged with non-zero factors (float32)
    m = attach_lorma(copy.deepcopy(base), targets, rank=4, alpha=8)
    for n, p in m.named_parameters():
        if "lorma_A" in n: nn.init.normal_(p, std=0.2)
        if "lorma_B" in n: nn.init.normal_(p, std=0.2)
    m.eval(); un = m(ids).detach(); mg = merge_lorma_(copy.deepcopy(m)); mgo = mg(ids).detach()
    assert not any(isinstance(x, LoRMALinear) for x in mg.modules())
    err = (un - mgo).abs().max().item(); eff = (un - h0).abs().max().item()
    assert err < 1e-3 and eff > 1e-2, (err, eff)
    print(f"tiny decoder: identity at init (both inits) | count == hand ({hand} params) | frozen weights untouched | gradient pattern | merged==unmerged (max err {err:.1e}, update effect {eff:.2f})  -> PASS")
tiny_model_tests()

_hand = lorma_param_count(cfg, ALL_TARGETS, 8); _hand_attn = lorma_param_count(cfg, ("q", "k", "v", "o"), 8)
print(f"hand-derived LoRMA parameters at r=8: attn(q,k,v,o) = {_hand_attn:,} | + MLP(gate,up,down) = {_hand:,}  ({_hand/BACKBONE_PARAMS:.2%} of the backbone)")

,swap_ab,a_std,out_err,max_grad_err,init_err,grad_pattern_ok,rank_A,rank_B,rank_naive_BA,rank_inflated_I_plus_sBA,rank_update_W_minus_W0
0,False,1.00,0.0,0.0,0.0,True,3,3,3,16,3
1,False,0.25,0.0,0.0,0.0,True,3,3,3,16,3
2,True,1.00,0.0,0.0,0.0,True,3,3,3,16,3
3,True,0.25,0.0,0.0,0.0,True,3,3,3,16,3


float64: dense == re-ordered (outputs AND gradients), identity at init, correct step-0 gradient pattern, rank(BA)<=r, rank(I+sBA)=d_out, rank(W'-W0)<=r  -> PASS
relative output change after one Adam step (lr 1e-3) of B: {'a_std=1.0000': '14.93%', 'a_std=0.0361': '0.54%'}
tiny decoder: identity at init (both inits) | count == hand (7168 params) | frozen weights untouched | gradient pattern | merged==unmerged (max err 1.1e-06, update effect 1.49)  -> PASS
hand-derived LoRMA parameters at r=8: attn(q,k,v,o) = 524,288 | + MLP(gate,up,down) = 1,769,472  (1.27% of the backbone)


## 7. Document representation (fixed by Experiment 1): two windows per case

(When the decoder-specific pooling variant of probe 2 is active, a closing `</s>` is appended to every window and the content is trimmed by one token so the window stays ≤ 512.)

Each case is cut into **`head`** (first 511 content tokens) and **`cue_rest`** (sentences carrying `<SECTION>` / `<ACT>` / "Section" / "Article" cues from the part of the case *beyond* the head window, kept in document order; the best 2-window variant in Exp. 1). Every window is prefixed with `</s>` and encoded independently by the frozen decoder; the token states of both windows are pooled jointly. Windows that would be empty (short cases) are marked invalid and ignored.

In [11]:
MAX_SEG = 96            # long "sentences" are cut into pieces of at most this many tokens
CUE_CAP = 3             # a segment's cue score saturates at this many cue words
LAST_BONUS_TOKENS = 128 # segments inside the last 128 tokens get a small bonus (operative part)

def select_cue_tokens(tok, fl, budget=CONTENT, head_keep=128):
    """Relevance-based context selection. tok/fl: uint16/uint8 arrays of the (sub)document."""
    n = len(tok)
    if n <= budget: return tok
    head = tok[:head_keep]; rest = tok[head_keep:]; rfl = fl[head_keep:]; budget_rest = budget - len(head)
    nr = len(rest)
    ends = np.flatnonzero(rfl & FLAG_SENT) + 1
    bounds = [0] + ends.tolist()
    if bounds[-1] != nr: bounds.append(nr)
    starts_l, ends_l = [], []
    for a, b in zip(bounds[:-1], bounds[1:]):
        while b - a > MAX_SEG:
            starts_l.append(a); ends_l.append(a + MAX_SEG); a += MAX_SEG
        if b > a: starts_l.append(a); ends_l.append(b)
    starts = np.asarray(starts_l); ends_a = np.asarray(ends_l)
    cs = np.concatenate([[0], np.cumsum((rfl & FLAG_CUE).astype(np.int64))])
    score = np.minimum(cs[ends_a] - cs[starts], CUE_CAP).astype(np.float64)
    score += 0.5 * (starts >= nr - LAST_BONUS_TOKENS)
    score -= np.arange(len(starts)) * 1e-6                       # stable tie-break: earlier first
    order = np.argsort(-score, kind="stable")
    chosen, used = [], 0                                    # chosen: list of (start, end) inside `rest`
    for k in order:
        s0, e0 = int(starts[k]), int(ends_a[k]); ln = e0 - s0; room = budget_rest - used
        if ln <= room:
            chosen.append((s0, e0)); used += ln
        elif room >= 16:                                    # truncate one last segment to use the remaining room
            chosen.append((s0, s0 + room)); used += room
        if budget_rest - used < 8: break
    chosen.sort()
    pieces = [rest[s0:e0] for s0, e0 in chosen]
    return np.concatenate([head] + pieces) if pieces else head

def make_view(name, tok, fl):
    n = len(tok)
    if name == "head":      return tok[:CONTENT]
    if name == "tail":      return tok[max(CONTENT, n - CONTENT):] if n > CONTENT else tok[:0]
    if name == "headtail":
        if n <= CONTENT: return tok
        h = CONTENT // 2; return np.concatenate([tok[:h], tok[n - (CONTENT - h):]])
    if name == "cue":       return select_cue_tokens(tok, fl, CONTENT, head_keep=128)
    if name == "cue_rest":
        if n <= CONTENT: return tok[:0]
        return select_cue_tokens(tok[CONTENT:], fl[CONTENT:], CONTENT, head_keep=0)
    raise ValueError(name)

class LSIDataset(Dataset):
    def __init__(self, split, indices, views, with_labels=True, append_end=False):
        self.a = ARR[split]; self.indices = np.asarray(indices, dtype=np.int64); self.views = tuple(views)
        self.with_labels = with_labels; self.append_end = append_end; self.need_flags = any(v.startswith("cue") for v in views)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        d = int(self.indices[i]); a = self.a; o0, o1 = int(a["offsets"][d]), int(a["offsets"][d + 1])
        tok = np.asarray(a["tokens"][o0:o1]); fl = np.asarray(a["flags"][o0:o1]) if self.need_flags else None
        seqs, valid = [], []
        for v in self.views:
            c = make_view(v, tok, fl)
            if self.append_end: c = c[:CONTENT - 1]          # keep prefix + content + closing </s> within 512 tokens
            seqs.append(np.concatenate([[PREFIX_ID], c.astype(np.int64)] + ([[EOS_ID]] if self.append_end else []))); valid.append(len(c) > 0)
        valid[0] = True
        item = {"seqs": seqs, "valid": valid, "idx": d}
        if self.with_labels: item["y"] = a["labels"][d].astype(np.float32)
        return item

def collate_lsi(batch):
    B, V = len(batch), len(batch[0]["seqs"]); T = max(len(s) for b in batch for s in b["seqs"])
    ids = torch.full((B, V, T), PAD_ID, dtype=torch.long)
    for i, b in enumerate(batch):
        for v, s in enumerate(b["seqs"]): ids[i, v, :len(s)] = torch.from_numpy(s)
    out = {"input_ids": ids, "mask": ids.ne(PAD_ID), "valid": torch.tensor([b["valid"] for b in batch]),
           "idx": torch.tensor([b["idx"] for b in batch])}
    if "y" in batch[0]: out["y"] = torch.from_numpy(np.stack([b["y"] for b in batch]))
    return out

# ---- sanity check of views on real documents
_chk = np.random.default_rng(0).choice(len(ARR["train"]["offsets"]) - 1, 300, replace=False)
stat = {k: [] for k in ["head", "tail", "headtail", "cue", "cue_rest"]}
cue_cnt = {k: [] for k in stat}
for d in _chk:
    a = ARR["train"]; o0, o1 = a["offsets"][d], a["offsets"][d + 1]
    tok, fl = np.asarray(a["tokens"][o0:o1]), np.asarray(a["flags"][o0:o1])
    if len(tok) <= CONTENT: continue
    for k in stat:
        v = make_view(k, tok, fl); stat[k].append(len(v))
        # cue density: use flags of the *original positions* is not available after cutting, so recount via a second selection pass
    cue_cnt["head"].append(int((fl[:CONTENT] & FLAG_CUE).sum()))
    cue_cnt["tail"].append(int((fl[max(CONTENT, len(tok) - CONTENT):] & FLAG_CUE).sum()))
print("lengths (docs longer than one window) :", {k: f"{np.mean(v):.0f}" for k, v in stat.items() if v})
print("cue words inside head view / tail view:", f"{np.mean(cue_cnt['head']):.1f} / {np.mean(cue_cnt['tail']):.1f}")
_demo = LSIDataset("train", [int(_chk[0])], ("head", "cue_rest"))[0]
print("example item: views", [len(s) for s in _demo["seqs"]], "valid", _demo["valid"], "labels", int(_demo["y"].sum()))
assert all(len(s) <= MAX_LEN for s in _demo["seqs"])

lengths (docs longer than one window) : {'head': '511', 'tail': '418', 'headtail': '511', 'cue': '509', 'cue_rest': '416'}
cue words inside head view / tail view: 3.4 / 2.6
example item: views [512, 512] valid [True, True] labels 1


## 6. Evaluation + post-processing toolkit (NumPy; identical to Experiment 1)

* `best_global` – one threshold shared by all statutes, picked by macro-F1.
* `fit_per_label` – a threshold per statute on a smoothed F1-vs-threshold curve, **shrunk toward the global threshold** in proportion to the statute's support (rare statutes would otherwise get noisy thresholds).
* `predict_binary(..., ensure_one=True)` – a document never receives an empty label set (best-ratio label is used).
* Calibration: temperature scaling and per-label Platt scaling. Label co-occurrence: a regularised linear *stacker* `z' = a⊙z + zW + b` that re-scores each statute from all 100 logits.
* `crossfit(...)` – every post-processing option is scored **out-of-fold** (K-fold), so fitting thresholds on dev does not inflate the comparison.

In [12]:
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import average_precision_score

GRID = np.round(np.arange(0.01, 0.96, 0.01), 2)
sigmoid = lambda z: 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def f1_grid(y, p, grid=GRID):
    y = y.astype(bool); npos = y.sum(0); G, C = len(grid), y.shape[1]
    tp = np.empty((G, C)); fp = np.empty((G, C))
    for gi, g in enumerate(grid):
        pr = p >= g; tp[gi] = (pr & y).sum(0); fp[gi] = pr.sum(0) - tp[gi]
    fn = npos[None, :] - tp
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def best_global(y, p, grid=GRID):
    f1 = f1_grid(y, p, grid); macro = f1.mean(1); i = int(np.argmax(macro))
    return float(grid[i]), float(macro[i])

def fit_per_label(y, p, t_glob, min_pos=10, shrink=100.0, grid=GRID):
    f1 = uniform_filter1d(f1_grid(y, p, grid), size=3, axis=0, mode="nearest")
    thr = grid[np.argmax(f1, axis=0)].astype(np.float64)
    sup = y.sum(0).astype(np.float64)
    thr = np.where(sup < min_pos, t_glob, thr)
    w = sup / (sup + shrink) if shrink > 0 else np.ones_like(sup)
    return w * thr + (1 - w) * t_glob

def predict_binary(p, thr, ensure_one=True):
    pred = p >= thr
    if ensure_one:
        empty = ~pred.any(1)
        if empty.any(): pred[empty, np.argmax(p[empty] / thr[None, :] if np.ndim(thr) else p[empty], axis=1)] = True
    return pred.astype(np.uint8)

def label_f1(y, pred):
    y = y.astype(bool); pr = pred.astype(bool); tp = (y & pr).sum(0); fp = (~y & pr).sum(0); fn = (y & ~pr).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def prf_summary(y, pred):
    yb, pb = y.astype(bool), pred.astype(bool)
    tp = (yb & pb).sum(0); fp = (~yb & pb).sum(0); fn = (yb & ~pb).sum(0)
    prec = tp / np.maximum(tp + fp, 1); rec = tp / np.maximum(tp + fn, 1)
    return dict(macro_f1=float(label_f1(y, pred).mean()),
                micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
                macro_precision=float(prec.mean()), macro_recall=float(rec.mean()),
                subset_acc=float((yb == pb).all(1).mean()), avg_pred=float(pb.sum(1).mean()),
                empty_rate=float((pb.sum(1) == 0).mean()))

# ---------------- calibration / co-occurrence re-scoring (all operate on LOGITS) ----------------
def fit_temperature(y, z):
    best, bt = 1e9, 1.0
    for T in np.arange(0.5, 4.01, 0.1):
        p = np.clip(sigmoid(z / T), 1e-7, 1 - 1e-7)
        l = -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
        if l < best: best, bt = l, T
    def tf(z2, T=bt): return z2 / T
    tf.params = {"type": "temperature", "T": float(bt)}
    return tf

def fit_platt(y, z, lam=1.0, iters=30):
    C = y.shape[1]; a = np.ones(C); b = np.zeros(C)
    for _ in range(iters):
        p = sigmoid(z * a + b); r = p - y; w = p * (1 - p)
        ga = (r * z).sum(0) + lam * (a - 1); gb = r.sum(0) + lam * b
        haa = (w * z * z).sum(0) + lam; hab = (w * z).sum(0); hbb = w.sum(0) + lam
        det = haa * hbb - hab ** 2 + 1e-12
        a -= (hbb * ga - hab * gb) / det; b -= (-hab * ga + haa * gb) / det
        a = np.clip(a, 0.05, 5.0)
    def tf(z2, a=a, b=b): return z2 * a + b
    tf.params = {"type": "platt", "a": a.tolist(), "b": b.tolist()}
    return tf

def fit_stacker(y, z, l2=0.01, steps=100, lr=0.02, seed=0):
    """z' = a*z + z@W + b  (W has zero diagonal). Adam on mean BCE (summed over labels) + 0.5*l2*||W||^2."""
    N, C = z.shape; rng = np.random.default_rng(seed)
    a = np.ones(C); b = np.zeros(C); W = np.zeros((C, C)); eye = 1 - np.eye(C)
    params = [a, b, W]; m = [np.zeros_like(q) for q in params]; v = [np.zeros_like(q) for q in params]
    zc = np.clip(z, -12, 12)
    for t in range(1, steps + 1):
        logit = zc * a + zc @ W + b; g = (sigmoid(logit) - y) / N
        ga = (g * zc).sum(0); gb = g.sum(0); gW = (zc.T @ g) * eye + l2 * W
        for k, gr in enumerate([ga, gb, gW]):
            m[k] = 0.9 * m[k] + 0.1 * gr; v[k] = 0.999 * v[k] + 0.001 * gr ** 2
            params[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
        W *= eye
    def tf(z2, a=a, b=b, W=W): return np.clip(z2, -12, 12) * a + np.clip(z2, -12, 12) @ W + b
    tf.params = {"type": "stacker", "a": a.tolist(), "b": b.tolist(), "W": W.tolist()}
    return tf

def stack_oof(y, z, l2, K=3):
    idx = np.random.default_rng(1).permutation(len(z)); folds = np.array_split(idx, K); out = np.zeros_like(z)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k])
        out[te] = fit_stacker(y[tr], z[tr], l2)(z[te])
    return out

def make_strategy(calib=None, thr="global", shrink=100.0, min_pos=10, top1=True, l2=0.01):
    def fit(y, z):
        if calib == "temp":    tf = fit_temperature(y, z); z_thr = tf(z)
        elif calib == "platt": tf = fit_platt(y, z);       z_thr = tf(z)
        elif calib == "stack": tf = fit_stacker(y, z, l2); z_thr = stack_oof(y, z, l2)   # out-of-fold logits for thresholds
        else:
            def tf(q): return q
            tf.params = {"type": "identity"}; z_thr = z
        p_tr = sigmoid(z_thr); tg, _ = best_global(y, p_tr)
        thr_vec = np.full(y.shape[1], tg) if thr == "global" else fit_per_label(y, p_tr, tg, min_pos, shrink)
        def predict(z2): return predict_binary(sigmoid(tf(z2)), thr_vec, top1)
        predict.thr = thr_vec; predict.tf = tf; return predict
    return fit

STRATEGIES = OrderedDict([
    ("global",                         make_strategy(None,    "global", top1=False)),
    ("global + top1",                  make_strategy(None,    "global")),
    ("per-label raw + top1",           make_strategy(None,    "label", shrink=0)),
    ("per-label shrunk(100) + top1",   make_strategy(None,    "label", shrink=100)),
    ("per-label shrunk(300) + top1",   make_strategy(None,    "label", shrink=300)),
    ("temperature -> global",          make_strategy("temp",  "global")),
    ("Platt -> global",                make_strategy("platt", "global")),
    ("Platt -> per-label shrunk(100)", make_strategy("platt", "label", shrink=100)),
    ("stacker -> global",              make_strategy("stack", "global")),
    ("stacker -> per-label shrunk(100)", make_strategy("stack", "label", shrink=100)),
])

def crossfit(y, z, fit_fn, K=5, seed=0):
    rng = np.random.default_rng(seed); folds = np.array_split(rng.permutation(len(z)), K); pred = np.zeros(y.shape, np.uint8)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k])
        pred[te] = fit_fn(y[tr], z[tr])(z[te])
    return pred

def ece_brier(y, p, bins=15):
    pf, yf = p.ravel(), y.ravel().astype(float); edges = np.linspace(0, 1, bins + 1); ece = 0.0
    idx = np.clip(np.digitize(pf, edges) - 1, 0, bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): ece += m.mean() * abs(pf[m].mean() - yf[m].mean())
    return float(ece), float(((pf - yf) ** 2).mean())

def macro_ap(y, p):
    v = [average_precision_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if 0 < y[:, j].sum() < len(y)]
    return float(np.mean(v)) if v else 0.0

def evaluate_logits(y, z, perlabel_cv=True):
    """Metrics used to rank experiments. Primary = macro-F1 at the best single global threshold."""
    p = sigmoid(z); thr, mf1 = best_global(y, p); pred = (p >= thr).astype(np.uint8)
    out = prf_summary(y, pred); out["best_global_thr"] = thr; out["macro_f1_global"] = mf1
    pc = np.clip(p, 1e-7, 1 - 1e-7); out["bce"] = float(-(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean())
    out["macro_ap"] = macro_ap(y, p)
    if perlabel_cv:
        pr = crossfit(y, z, STRATEGIES["per-label shrunk(100) + top1"], K=4)
        out["macro_f1_perlabel_cv"] = float(label_f1(y, pr).mean())
    return out

# ---- self-test on synthetic correlated multi-label data
_rng = np.random.default_rng(0); _N, _C = 3000, 12
_lat = _rng.normal(size=(_N, 4)); _W = _rng.normal(size=(4, _C)); _s = _lat @ _W - 1.5
_y = (_rng.random((_N, _C)) < sigmoid(_s)).astype(np.uint8); _z = _s + _rng.normal(scale=1.2, size=_s.shape)
_tmp = evaluate_logits(_y, _z); print("self-test:", {k: round(v, 3) for k, v in _tmp.items() if k in ["macro_f1_global", "macro_ap", "macro_f1_perlabel_cv"]})

self-test: {'macro_f1_global': 0.552, 'macro_ap': 0.558, 'macro_f1_perlabel_cv': 0.553}


## 8. Classification head, loss and the LoRMA classification model

In [13]:
D_MODEL = cfg.d_model
def content_mask(ids, pool_mask):
    return pool_mask & ids.ne(BOS_ID) & ids.ne(EOS_ID)

class MeanHead(nn.Module):          # masked mean over content tokens (BOS/EOS/PAD excluded)
    def __init__(self, C, dropout):
        super().__init__(); self.drop = nn.Dropout(dropout); self.out = nn.Linear(D_MODEL, C)
    def bias_param(self): return self.out.bias
    def forward(self, h, ids, pm, last=None):
        cm = content_mask(ids, pm).unsqueeze(-1).to(h.dtype)
        return self.out(self.drop((h * cm).sum(1) / cm.sum(1).clamp_min(1.0)))

class AttnPoolHead(nn.Module):      # one learned query attends over content tokens
    def __init__(self, C, dropout, d_att=256):
        super().__init__(); self.proj = nn.Linear(D_MODEL, d_att); self.v = nn.Linear(d_att, 1, bias=False)
        self.drop = nn.Dropout(dropout); self.out = nn.Linear(D_MODEL, C)
    def bias_param(self): return self.out.bias
    def forward(self, h, ids, pm, last=None):
        cm = content_mask(ids, pm)
        s = self.v(torch.tanh(self.proj(h))).squeeze(-1).float().masked_fill(~cm, -1e4)
        a = torch.softmax(s, dim=1) * cm; a = a / a.sum(1, keepdim=True).clamp_min(1e-9)
        return self.out(self.drop((a.unsqueeze(-1).to(h.dtype) * h).sum(1)))

class MeanMaxMLPHead(nn.Module):    # [mean ; max] -> LayerNorm -> MLP
    def __init__(self, C, dropout, hidden=512):
        super().__init__(); self.norm = nn.LayerNorm(2 * D_MODEL); self.fc = nn.Linear(2 * D_MODEL, hidden)
        self.drop = nn.Dropout(dropout); self.out = nn.Linear(hidden, C)
    def bias_param(self): return self.out.bias
    def forward(self, h, ids, pm, last=None):
        cmb = content_mask(ids, pm); cm = cmb.unsqueeze(-1).to(h.dtype)
        mean = (h * cm).sum(1) / cm.sum(1).clamp_min(1.0)
        mx = h.masked_fill(~cmb.unsqueeze(-1), -1e4).max(1).values
        mx = torch.where(cmb.any(1, keepdim=True), mx, torch.zeros_like(mx))
        return self.out(self.drop(F.gelu(self.fc(self.norm(torch.cat([mean, mx], -1))))))

class LabelAttnHead(nn.Module):     # LAAT-style: every statute has its own attention query over the tokens
    def __init__(self, C, dropout, d_att=256):
        super().__init__(); self.proj = nn.Linear(D_MODEL, d_att)
        self.Q = nn.Parameter(torch.randn(C, d_att) * 0.02); self.W = nn.Parameter(torch.randn(C, D_MODEL) * 0.02)
        self.bias = nn.Parameter(torch.zeros(C)); self.drop = nn.Dropout(dropout)
    def bias_param(self): return self.bias
    def forward(self, h, ids, pm, last=None):
        cm = content_mask(ids, pm)
        u = torch.tanh(self.proj(h))                                              # [N,T,d_att]
        s = torch.einsum("ntk,ck->nct", u.float(), self.Q.float()).masked_fill(~cm[:, None, :], -1e4)
        a = torch.softmax(s, dim=-1) * cm[:, None, :]; a = a / a.sum(-1, keepdim=True).clamp_min(1e-9)
        ctx = torch.einsum("nct,ntd->ncd", a, h.float())                          # [N,C,D]
        return (self.drop(ctx) * self.W.float()).sum(-1) + self.bias.float()

class AttnLastHead(nn.Module):
    """Decoder-specific: [attention-pooled token states ; state of the closing </s>] -> LayerNorm -> dropout -> linear.
    In a causal model only the LAST position of a window has attended to the whole window; LoRMA can learn to store a summary there."""
    def __init__(self, C, dropout, d_att=256):
        super().__init__(); self.proj = nn.Linear(D_MODEL, d_att); self.v = nn.Linear(d_att, 1, bias=False)
        self.norm = nn.LayerNorm(2 * D_MODEL); self.drop = nn.Dropout(dropout); self.out = nn.Linear(2 * D_MODEL, C)
    def bias_param(self): return self.out.bias
    def forward(self, h, ids, pm, last=None):
        if last is None: raise ValueError("AttnLastHead needs the last-token state")
        cm = content_mask(ids, pm)
        s = self.v(torch.tanh(self.proj(h))).squeeze(-1).float().masked_fill(~cm, -1e4)
        a = torch.softmax(s, dim=1) * cm; a = a / a.sum(1, keepdim=True).clamp_min(1e-9)
        pooled = (a.unsqueeze(-1).to(h.dtype) * h).sum(1)
        return self.out(self.drop(self.norm(torch.cat([pooled, last.to(pooled.dtype)], dim=-1))))

HEADS = {"mean": MeanHead, "attn": AttnPoolHead, "attn_last": AttnLastHead, "meanmax": MeanMaxMLPHead, "label_attn": LabelAttnHead}   # (the <s>-pooling head is meaningless for a causal model)

class LSIModel(nn.Module):
    """Frozen causal decoder (+ LoRMA adapters)  ->  token states of every window  ->  pooling head  ->  100 statute logits."""
    def __init__(self, backbone, head, n_views, view_agg, dropout, prior_logit):
        super().__init__()
        self.backbone, self.view_agg, self.n_views = backbone, view_agg, n_views
        self.head = HEADS[head](NUM_LABELS, dropout)
        with torch.no_grad(): self.head.bias_param().copy_(torch.as_tensor(prior_logit, dtype=torch.float32))   # start at the label prior
    def forward(self, input_ids, mask, valid):
        B, V, T = input_ids.shape
        h = self.backbone(input_ids.reshape(B * V, T)).float()          # causal attention: right padding never affects real tokens
        pm = mask & valid[:, :, None]
        lengths = mask.reshape(B * V, T).sum(1)                           # right-padded -> last real position = length - 1
        last_all = h[torch.arange(B * V, device=h.device), (lengths - 1).clamp_min(0)]      # [B*V, D] state of each window's final token
        if V == 1 or self.view_agg == "tokens":                           # pool over all tokens of all windows jointly
            vf = valid.float()[:, :, None]
            last = (last_all.reshape(B, V, -1) * vf).sum(1) / vf.sum(1).clamp_min(1.0)       # mean over valid windows
            return self.head(h.reshape(B, V * T, -1), input_ids.reshape(B, V * T), pm.reshape(B, V * T), last=last)
        lv = self.head(h, input_ids.reshape(B * V, T), pm.reshape(B * V, T), last=last_all).float().reshape(B, V, -1)
        lv = lv.masked_fill(~valid[:, :, None], -1e4)
        return torch.logsumexp(lv, dim=1) - torch.log(valid.sum(1).clamp_min(1).float())[:, None]

PRIOR_LOGIT = np.log(PRIOR / (1 - PRIOR)).astype(np.float32)

def make_loss(name):
    """Returns f(logits, targets) -> scalar. All inputs float32 [B, C]."""
    if name == "bce":
        return lambda z, y: F.binary_cross_entropy_with_logits(z, y)
    if name == "bce_posw":        # per-statute positive weight = sqrt(neg/pos), clipped to [1, 20]
        pw = torch.as_tensor(np.clip(np.sqrt((len(ARR["train"]["labels"]) - POS_COUNT) / POS_COUNT), 1, 20), dtype=torch.float32, device=device)
        return lambda z, y: F.binary_cross_entropy_with_logits(z, y, pos_weight=pw)
    if name == "focal":           # gamma = 2
        def f(z, y):
            ce = F.binary_cross_entropy_with_logits(z, y, reduction="none"); p = torch.sigmoid(z)
            pt = p * y + (1 - p) * (1 - y); return ((1 - pt) ** 2 * ce).mean()
        return f
    if name == "asl":             # Asymmetric Loss (Ridnik et al.): gamma+=0, gamma-=4, probability margin 0.05
        def f(z, y, g_neg=4.0, clip=0.05, eps=1e-8):
            p = torch.sigmoid(z); pm = (p - clip).clamp(min=0)
            lp = y * torch.log(p.clamp(min=eps)); ln = (1 - y) * (pm ** g_neg) * torch.log((1 - pm).clamp(min=eps))
            return -(lp + ln).mean()
        return f
    raise ValueError(name)

# ---- shape test of every head on random features
for _name, _H in HEADS.items():
    _h = _H(NUM_LABELS, 0.1).to(device); _x = torch.randn(3, 40, D_MODEL, device=device)
    _ids = torch.randint(4, 100, (3, 40), device=device); _ids[:, 0] = BOS_ID; _ids[:, -1] = EOS_ID
    _pm = torch.ones(3, 40, dtype=torch.bool, device=device); _pm[2, 20:] = False
    _o = _h(_x, _ids, _pm, last=torch.randn(3, D_MODEL, device=device)); assert _o.shape == (3, NUM_LABELS) and torch.isfinite(_o).all(), _name
    del _h, _x, _ids, _pm, _o
for _l in ["bce", "bce_posw", "focal", "asl"]:
    _v = make_loss(_l)(torch.randn(4, NUM_LABELS, device=device), (torch.rand(4, NUM_LABELS, device=device) < 0.03).float()); assert torch.isfinite(_v), _l
print("heads + losses OK:", list(HEADS), ["bce", "bce_posw", "focal", "asl"])

heads + losses OK: ['mean', 'attn', 'attn_last', 'meanmax', 'label_attn'] ['bce', 'bce_posw', 'focal', 'asl']


## 9. Training engine

* Every run: fresh `LegalDecoder` ← Phase 1 weights, **everything frozen**, LoRMA₊ wrappers attached to the chosen matrices, fresh attention-pooling head (bias at the label prior).
* Trainable = LoRMA factors (`lorma_lr`) + head (`head_lr`) only. AdamW, linear warm-up → linear decay, fp16 AMP, grad-clip 1.0.
* Frozen weights need no weight-gradients and no optimizer state, so this is cheaper than the full fine-tuning of Exp. 1; the micro-batch / gradient-checkpointing choice is auto-tuned in the next cell.

In [14]:
GRAD_CKPT, MB_SEQS, NUM_WORKERS = True, 16, 2
MAX_GRAD_NORM = 1.0

def mb_docs_for(rc):  return max(1, MB_SEQS // len(rc["views"]))
def accum_for(rc):    return max(1, EFFECTIVE_BATCH // mb_docs_for(rc))

def build_model(rc, grad_ckpt=None):
    dec = LegalDecoder(cfg, grad_ckpt=GRAD_CKPT if grad_ckpt is None else grad_ckpt)
    load_gqa_weights(dec, GQA_STATE)
    attach_lorma(dec, tuple(rc["targets"]), rank=rc["rank"], alpha=rc["alpha"], swap_ab=rc.get("swap_ab", False), a_std=rc.get("a_std"))
    return LSIModel(dec, rc["head"], len(rc["views"]), rc["view_agg"], rc["dropout"], PRIOR_LOGIT).to(device)

def is_trainable_name(n): return ("lorma_" in n) or n.startswith("head.")
def trainable_state(model): return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if is_trainable_name(k)}

def make_optimizer(model, rc):
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        lr = rc["lorma_lr"] if "lorma_" in name else rc["head_lr"]
        wd = 0.0 if p.ndim <= 1 else rc["wd"]
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()], betas=(0.9, 0.999))

def build_training(model, rc, n_docs, epochs):
    mb, accum = mb_docs_for(rc), accum_for(rc)
    steps_per_epoch = math.ceil(math.ceil(n_docs / mb) / accum); total = steps_per_epoch * epochs
    warm = max(1, int(total * rc["warmup"]))
    lam = lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))
    opt = make_optimizer(model, rc)
    return dict(opt=opt, sched=torch.optim.lr_scheduler.LambdaLR(opt, lam), scaler=torch.amp.GradScaler("cuda"), mb=mb, accum=accum,
                total_steps=total, steps_per_epoch=steps_per_epoch, loss_fn=make_loss(rc["loss"]),
                params=[p for p in model.parameters() if p.requires_grad])

def make_loader(ds, batch, shuffle, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=g if shuffle else None, collate_fn=collate_lsi,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

def to_dev(b):
    return (b["input_ids"].to(device, non_blocking=True), b["mask"].to(device, non_blocking=True), b["valid"].to(device, non_blocking=True))

def train_one_epoch(model, loader, T, tag="", log_frac=0.25):
    model.train(); opt, sched, scaler, loss_fn, accum = T["opt"], T["sched"], T["scaler"], T["loss_fn"], T["accum"]
    opt.zero_grad(set_to_none=True); tot, nd, t0, n_micro = 0.0, 0, time.perf_counter(), len(loader)
    next_log = max(1, int(n_micro * log_frac))
    for i, b in enumerate(loader, 1):
        ids, mask, valid = to_dev(b); y = b["y"].to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(ids, mask, valid)
        loss = loss_fn(logits.float(), y)
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite loss at micro-batch {i} ({tag})")
        scaler.scale(loss / accum).backward()
        tot += loss.item() * ids.shape[0]; nd += ids.shape[0]
        if i % accum == 0 or i == n_micro:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(T["params"], MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        if i >= next_log or i == n_micro:
            next_log += max(1, int(n_micro * log_frac))
            print(f"   {tag} {i}/{n_micro} | loss {tot/nd:.4f} | {nd/(time.perf_counter()-t0):.1f} docs/s", flush=True)
    return tot / max(nd, 1), nd / (time.perf_counter() - t0)

@torch.inference_mode()
def predict_logits(model, ds, batch_docs, desc="", amp=True):
    model.eval(); out = []; t0 = time.perf_counter()
    for b in make_loader(ds, batch_docs, False):
        ids, mask, valid = to_dev(b)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            out.append(model(ids, mask, valid).float().cpu().numpy())
    z = np.concatenate(out); return z, len(ds) / (time.perf_counter() - t0)
print("engine ready")

engine ready


### 9b. Auto-tune micro-batch / gradient checkpointing for this GPU (real recipe: 2 windows, all 7 LoRMA targets)

In [15]:
def autotune_memory():
    global GRAD_CKPT, MB_SEQS
    rc = dict(BASE); V = len(rc["views"]); res = []
    for ckpt, seqs in [(False, 32), (False, 16), (True, 32), (True, 16), (True, 8)]:
        model = opt = scaler = None
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model = build_model(rc, grad_ckpt=ckpt); model.train(); opt = make_optimizer(model, rc); scaler = torch.amp.GradScaler("cuda")
            B = seqs // V
            ids = torch.randint(4, 50_000, (B, V, MAX_LEN), device=device); ids[..., 0] = PREFIX_ID
            mask = torch.ones_like(ids, dtype=torch.bool); valid = torch.ones(B, V, dtype=torch.bool, device=device)
            y = (torch.rand(B, NUM_LABELS, device=device) < 0.03).float()
            for step in range(4):
                if step == 1: torch.cuda.synchronize(); t0 = time.perf_counter()
                with torch.autocast("cuda", dtype=torch.float16): lg = model(ids, mask, valid)
                loss = F.binary_cross_entropy_with_logits(lg.float(), y); opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            torch.cuda.synchronize(); sps = seqs * 3 / (time.perf_counter() - t0)
            res.append((sps, ckpt, seqs, torch.cuda.max_memory_allocated() / 1024**3))
            print(f"grad_ckpt={ckpt!s:5} seqs/micro-batch={seqs:2d}: {sps:5.1f} seq/s | peak {res[-1][3]:.1f} GiB")
        except torch.cuda.OutOfMemoryError:
            print(f"grad_ckpt={ckpt!s:5} seqs/micro-batch={seqs:2d}: OOM")
        finally:
            del model, opt, scaler; gc.collect(); torch.cuda.empty_cache()
    if not res: raise RuntimeError("Nothing fits in GPU memory.")
    sps, GRAD_CKPT, MB_SEQS, _ = max(res)
    print(f"-> using grad_ckpt={GRAD_CKPT}, {MB_SEQS} sequences per micro-batch ({sps:.1f} seq/s on this GPU)")
    return sps

SEQ_PER_S = autotune_memory()
def minutes_for(n_docs, n_views, epochs): return n_docs * epochs * n_views / SEQ_PER_S / 60
_ntr = len(ARR["train"]["labels"]); _ndv = len(ARR["dev"]["labels"]); _V = len(BASE["views"])
_nprobe = len(PROBE_GRID) + (1 if RUN_POOL_PROBE else 0)
print(f"estimated probes: {_nprobe} runs x {minutes_for(EXP_TRAIN_DOCS, _V, EXP_EPOCHS):.0f} min = {_nprobe*minutes_for(EXP_TRAIN_DOCS, _V, EXP_EPOCHS)/60:.1f} h (+ eval)")
print(f"estimated final: {minutes_for(_ntr, _V, 1):.0f} min per epoch (+ ~{minutes_for(_ndv, _V, 1)/2.5:.0f} min dev eval)  ->  up to {FINAL_MAX_EPOCHS} epochs = {FINAL_MAX_EPOCHS*(minutes_for(_ntr, _V, 1)+minutes_for(_ndv, _V, 1)/2.5)/60:.1f} h (early stopping may end earlier)")

grad_ckpt=False seqs/micro-batch=32:  31.9 seq/s | peak 10.4 GiB
grad_ckpt=False seqs/micro-batch=16:  31.0 seq/s | peak 5.6 GiB
grad_ckpt=True  seqs/micro-batch=32:  23.1 seq/s | peak 2.1 GiB
grad_ckpt=True  seqs/micro-batch=16:  22.5 seq/s | peak 1.3 GiB
grad_ckpt=True  seqs/micro-batch= 8:  21.4 seq/s | peak 1.0 GiB
-> using grad_ckpt=False, 32 sequences per micro-batch (31.9 seq/s on this GPU)
estimated probes: 4 runs x 17 min = 1.1 h (+ eval)
estimated final: 45 min per epoch (+ ~4 min dev eval)  ->  up to 6 epochs = 4.9 h (early stopping may end earlier)


## 10. Probe subsets + experiment runner (results persisted → safe to re-run after a crash)

In [16]:
Ntr, Ndv, Nte = len(ARR["train"]["labels"]), len(ARR["dev"]["labels"]), len(ARR["test"]["ids"])
rng = np.random.default_rng(SEED)
exp_train_idx = rng.permutation(Ntr)[:EXP_TRAIN_DOCS]
chosen = set(exp_train_idx.tolist())
for j in range(NUM_LABELS):                                   # make sure rare statutes are represented in the subset
    have = int(Ytr[exp_train_idx, j].sum())
    if have < MIN_POS_PER_LABEL_SUBSET:
        pool = [i for i in np.flatnonzero(Ytr[:, j]) if i not in chosen]
        add = rng.choice(pool, size=min(len(pool), MIN_POS_PER_LABEL_SUBSET - have), replace=False)
        chosen.update(int(a) for a in add)
exp_train_idx = np.array(sorted(chosen)); exp_dev_idx = np.sort(rng.permutation(Ndv)[:EXP_DEV_DOCS])
Y_dev_sub = Ydv[exp_dev_idx]
print(f"probe train docs: {len(exp_train_idx):,} | probe dev docs: {len(exp_dev_idx):,}")

def safe_name(n): return re.sub(r"[^A-Za-z0-9_.=+-]", "_", n)
def logits_path(n): return RUN_DIR / f"{safe_name(n)}_dev_logits.npy"
def rc_norm(rc):
    d = dict(rc); d["views"] = list(d["views"]); d["targets"] = list(d["targets"]); return d
def rc_key(rc): return json.dumps(rc_norm(rc), sort_keys=True, default=str)
def load_store(): return json.load(open(EXP_STORE)) if EXP_STORE.exists() else {}
def save_store(s):
    tmp = EXP_STORE.with_suffix(".tmp"); json.dump(s, open(tmp, "w"), indent=1, default=str); os.replace(tmp, EXP_STORE)

def run_experiment(name, rc, stage):
    rc = dict(rc); rc["views"] = tuple(rc["views"]); rc["targets"] = tuple(rc["targets"]); store = load_store(); key = rc_key(rc)
    for n, e in store.items():
        if e["rc_key"] == key and logits_path(n).exists():
            print(f"[reuse] {name}  (same recipe as '{n}')"); return dict(e, name=n)
    print("=" * 100); print(f"[run] {name} | targets={rc['targets']} lorma_lr={rc['lorma_lr']}")
    seed_everything(SEED); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = T = None
    try:
        model = build_model(rc); n_params = sum(p.numel() for p in model.parameters())
        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        assert n_train - sum(p.numel() for p in model.head.parameters()) == lorma_param_count(cfg, rc["targets"], rc["rank"]), "LoRMA parameter count != hand-derived"
        T = build_training(model, rc, len(exp_train_idx), EXP_EPOCHS)
        train_ds = LSIDataset("train", exp_train_idx, rc["views"], append_end=rc["end_token"]); dev_ds = LSIDataset("dev", exp_dev_idx, rc["views"], append_end=rc["end_token"])
        t0 = time.perf_counter(); dps = []
        for ep in range(1, EXP_EPOCHS + 1):
            loss, d = train_one_epoch(model, make_loader(train_ds, T["mb"], True, SEED + ep), T, tag=f"ep{ep}/{EXP_EPOCHS}"); dps.append(d)
            print(f"   epoch {ep} done | train loss {loss:.4f}")
        train_s = time.perf_counter() - t0
        z, eval_dps = predict_logits(model, dev_ds, T["mb"] * 2)
        m = evaluate_logits(Y_dev_sub, z)
        entry = dict(name=name, stage=stage, rc=rc_norm(rc), rc_key=key, **m, train_minutes=train_s / 60,
                     train_docs_per_s=float(np.mean(dps)), eval_docs_per_s=float(eval_dps), peak_gib=torch.cuda.max_memory_allocated() / 1024**3,
                     params_total=int(n_params), params_trainable=int(n_train), grad_ckpt=bool(GRAD_CKPT), mb_seqs=int(MB_SEQS))
        np.save(logits_path(name), z.astype(np.float32))
        store = load_store(); store[name] = entry; save_store(store)
        print(f"[done] {name}: macro-F1(global thr {m['best_global_thr']:.2f}) = {m['macro_f1_global']:.4f} | macro-AP {m['macro_ap']:.4f} | "
              f"per-label-CV {m['macro_f1_perlabel_cv']:.4f} | {train_s/60:.1f} min")
        return entry
    except torch.cuda.OutOfMemoryError:
        print(f"[OOM] {name}: lower MB_SEQS / EXP_TRAIN_DOCS or skip this variant"); return None
    finally:
        del model, T; gc.collect(); torch.cuda.empty_cache()

def recipe_cost(rc): return (len(rc["targets"]), len(rc["views"]), 0 if rc["head"] == "attn" else 1)
def choose_best(entries, tol=SELECTION_TOL):
    entries = [e for e in entries if e is not None]; top = max(e["macro_f1_global"] for e in entries)
    near = [e for e in entries if e["macro_f1_global"] >= top - tol]
    return min(near, key=lambda e: (recipe_cost(e["rc"]), -e["macro_f1_global"]))

def run_stage(stage, variants, base_rc):
    entries = []
    for label, ov in variants.items():
        e = run_experiment(f"{stage}:{label}", {**base_rc, **ov}, stage)
        if e is not None: entries.append(dict(e, label=label))
    best = choose_best(entries)
    df = pd.DataFrame([{"variant": e["label"], "macro_F1@global": e["macro_f1_global"], "thr": e["best_global_thr"], "macro_AP": e["macro_ap"],
                        "macro_F1 per-label(CV)": e["macro_f1_perlabel_cv"], "micro_F1": e["micro_f1"], "macro_P": e["macro_precision"],
                        "macro_R": e["macro_recall"], "subset_acc": e["subset_acc"], "avg_#labels": e["avg_pred"],
                        "trainable": e["params_trainable"], "train_min": e["train_minutes"], "docs/s": e["train_docs_per_s"], "peak_GiB": e["peak_gib"],
                        "selected": "<<<" if e["label"] == best["label"] else ""} for e in entries]).sort_values("macro_F1@global", ascending=False)
    print(f"\n######## {stage}: results (dev subset of {len(exp_dev_idx)} docs; parsimony tolerance {SELECTION_TOL}) ########")
    display(df.round(4).reset_index(drop=True))
    print(f"-> selected: '{best['label']}'"); return best, df

probe train docs: 8,057 | probe dev docs: 3,000


## 11. LoRMA probe — which matrices to adapt and which learning rate

These two choices did not exist in Experiment 1 (full fine-tuning has no adapter placement; its LR range does not transfer to a frozen-backbone adapter) so they are the *only* things tested here, on the 2-epoch subset: all seven matrices vs attention-only, at LR 3e-4 and 1e-3. Everything else is the Exp. 1 recipe. The winner (parsimony rule: among runs within 0.004 macro-F1 of the best, the one with fewer adapted matrices) is used for the final run.

In [17]:
BEST_RC = dict(BASE); PROBE_TABLE = None
if RUN_LORMA_PROBE:
    best, PROBE_TABLE = run_stage("PROBE", PROBE_GRID, BEST_RC)
    BEST_RC["targets"] = tuple(best["rc"]["targets"]); BEST_RC["lorma_lr"] = best["rc"]["lorma_lr"]
    lrs = sorted({v["lorma_lr"] for v in PROBE_GRID.values()})
    if BEST_RC["lorma_lr"] in (lrs[0], lrs[-1]) and len(lrs) > 1:
        print(f"note: the selected LR {BEST_RC['lorma_lr']:g} is at the edge of the probed range {lrs} - the optimum may lie beyond it")
print("\nFINAL RECIPE (Exp.1 recipe + LoRMA probe result):\n", json.dumps(rc_norm(BEST_RC), indent=1, default=str))
print("\nrecipe after probe 1:", {k: BEST_RC[k] for k in ("targets", "lorma_lr")})

[run] PROBE:all7 | lr 3e-4 | targets=('q', 'k', 'v', 'o', 'gate', 'up', 'down') lorma_lr=0.0003
   ep1/2 126/504 | loss 0.4630 | 15.1 docs/s
   ep1/2 252/504 | loss 0.4121 | 14.9 docs/s
   ep1/2 378/504 | loss 0.3883 | 14.8 docs/s
   ep1/2 504/504 | loss 0.3737 | 14.8 docs/s
   epoch 1 done | train loss 0.3737
   ep2/2 126/504 | loss 0.2938 | 14.6 docs/s
   ep2/2 252/504 | loss 0.2916 | 14.6 docs/s
   ep2/2 378/504 | loss 0.2904 | 14.6 docs/s
   ep2/2 504/504 | loss 0.2878 | 14.6 docs/s
   epoch 2 done | train loss 0.2878
[done] PROBE:all7 | lr 3e-4: macro-F1(global thr 0.46) = 0.2872 | macro-AP 0.2510 | per-label-CV 0.2888 | 18.3 min
[run] PROBE:all7 | lr 1e-3 | targets=('q', 'k', 'v', 'o', 'gate', 'up', 'down') lorma_lr=0.001
   ep1/2 126/504 | loss 0.4551 | 14.6 docs/s
   ep1/2 252/504 | loss 0.4075 | 14.6 docs/s
   ep1/2 378/504 | loss 0.3860 | 14.6 docs/s
   ep1/2 504/504 | loss 0.3723 | 14.6 docs/s
   epoch 1 done | train loss 0.3723
   ep2/2 126/504 | loss 0.2863 | 14.6 docs/s
 

,variant,macro_F1@global,thr,macro_AP,macro_F1 per-label(CV),micro_F1,macro_P,macro_R,subset_acc,avg_#labels,trainable,train_min,docs/s,peak_GiB,selected
0,all7 | lr 1e-3,0.2980,0.48,0.2588,0.3015,0.4088,0.2658,0.3624,0.0257,5.1057,2043492,18.3390,14.6447,10.4336,<<<
1,all7 | lr 3e-4,0.2872,0.46,0.2510,0.2888,0.4034,0.2506,0.3585,0.0220,5.3860,2043492,18.2782,14.6938,10.4336,
2,attn4 | lr 1e-3,0.2840,0.40,0.2447,0.2872,0.3935,0.2279,0.4032,0.0103,6.6843,798308,15.6232,17.1903,8.0272,


-> selected: 'all7 | lr 1e-3'
note: the selected LR 0.001 is at the edge of the probed range [0.0003, 0.001] - the optimum may lie beyond it

FINAL RECIPE (Exp.1 recipe + LoRMA probe result):
 {
 "views": [
  "head",
  "cue_rest"
 ],
 "view_agg": "tokens",
 "head": "attn",
 "loss": "bce_posw",
 "head_lr": 0.001,
 "dropout": 0.1,
 "warmup": 0.06,
 "wd": 0.01,
 "end_token": false,
 "rank": 8,
 "alpha": 16,
 "targets": [
  "q",
  "k",
  "v",
  "o",
  "gate",
  "up",
  "down"
 ],
 "lorma_lr": 0.001,
 "a_std": null
}

recipe after probe 1: {'targets': ('q', 'k', 'v', 'o', 'gate', 'up', 'down'), 'lorma_lr': 0.001}


## 11b. Probe 2 — decoder-specific pooling: attention pooling alone vs attention pooling + last-token state

A causal model only lets the **last** token of a window see the whole window. Variant B appends a closing `</s>` to every window (content trimmed by one token), takes its final hidden state (mean over the valid windows) and concatenates it with the attention-pooled vector before the classifier. Variant A is the Exp. 1 head. Both use the best LoRMA settings from probe 1 — variant A is therefore *re-used* from probe 1 (no extra training) when probe 1 ran, so this costs **one** extra run. The better macro-F1 wins; ties within 0.004 go to the simpler head A.

In [18]:
POOL_TABLE = None
if RUN_POOL_PROBE:
    best, POOL_TABLE = run_stage("POOL", POOL_PROBE_GRID, BEST_RC)
    BEST_RC["head"] = best["rc"]["head"]; BEST_RC["end_token"] = bool(best["rc"]["end_token"])
print("\nFINAL RECIPE (Exp.1 recipe + LoRMA probe + pooling probe):\n", json.dumps(rc_norm(BEST_RC), indent=1, default=str))
json.dump(rc_norm(BEST_RC), open(RUN_DIR / "final_recipe.json", "w"), indent=1, default=str)

[reuse] POOL:pool=attn (Exp.1 head)  (same recipe as 'PROBE:all7 | lr 1e-3')
[run] POOL:pool=attn + last-token (closing </s>) | targets=('q', 'k', 'v', 'o', 'gate', 'up', 'down') lorma_lr=0.001
   ep1/2 126/504 | loss 0.4395 | 14.6 docs/s
   ep1/2 252/504 | loss 0.3980 | 14.6 docs/s
   ep1/2 378/504 | loss 0.3802 | 14.6 docs/s
   ep1/2 504/504 | loss 0.3685 | 14.6 docs/s
   epoch 1 done | train loss 0.3685
   ep2/2 126/504 | loss 0.2827 | 14.6 docs/s
   ep2/2 252/504 | loss 0.2811 | 14.6 docs/s
   ep2/2 378/504 | loss 0.2794 | 14.6 docs/s
   ep2/2 504/504 | loss 0.2764 | 14.6 docs/s
   epoch 2 done | train loss 0.2764
[done] POOL:pool=attn + last-token (closing </s>): macro-F1(global thr 0.47) = 0.3065 | macro-AP 0.2695 | per-label-CV 0.3064 | 18.4 min

######## POOL: results (dev subset of 3000 docs; parsimony tolerance 0.004) ########


,variant,macro_F1@global,thr,macro_AP,macro_F1 per-label(CV),micro_F1,macro_P,macro_R,subset_acc,avg_#labels,trainable,train_min,docs/s,peak_GiB,selected
0,pool=attn + last-token (closing </s>),0.3065,0.47,0.2695,0.3064,0.4138,0.2673,0.3898,0.0210,5.4550,2123364,18.3514,14.6347,10.4348,<<<
1,pool=attn (Exp.1 head),0.2980,0.48,0.2588,0.3015,0.4088,0.2658,0.3624,0.0257,5.1057,2043492,18.3390,14.6447,10.4336,


-> selected: 'pool=attn + last-token (closing </s>)'

FINAL RECIPE (Exp.1 recipe + LoRMA probe + pooling probe):
 {
 "views": [
  "head",
  "cue_rest"
 ],
 "view_agg": "tokens",
 "head": "attn_last",
 "loss": "bce_posw",
 "head_lr": 0.001,
 "dropout": 0.1,
 "warmup": 0.06,
 "wd": 0.01,
 "end_token": true,
 "rank": 8,
 "alpha": 16,
 "targets": [
  "q",
  "k",
  "v",
  "o",
  "gate",
  "up",
  "down"
 ],
 "lorma_lr": 0.001,
 "a_std": null
}


## 12. FINAL TRAINING — full training set, **6 epochs, early stopping (patience 2) on dev**

Trains the LoRMA factors + head on all `train.json` documents; evaluates on the **full dev set** after every epoch (macro-F1 at the best global threshold); keeps the best epoch; stops after 2 epochs without improvement. Only the trainable tensors (LoRMA factors + head, a few MB) are checkpointed each epoch, so `RESUME_FINAL=True` can continue an interrupted run cheaply.

In [19]:
FINAL_RC = dict(BEST_RC); FINAL_RC["views"] = tuple(FINAL_RC["views"]); FINAL_RC["targets"] = tuple(FINAL_RC["targets"])
FINAL_BEST_PT, FINAL_LAST_PT = RUN_DIR / "final_best_trainable.pt", RUN_DIR / "final_last.pt"
FINAL_DEV_LOGITS = RUN_DIR / "final_best_dev_logits.npy"; FINAL_HISTORY = RUN_DIR / "final_history.json"

final_train_idx = np.arange(Ntr) if FINAL_TRAIN_DOCS_LIMIT is None else np.sort(np.random.default_rng(SEED).permutation(Ntr)[:FINAL_TRAIN_DOCS_LIMIT])
final_dev_idx = np.arange(Ndv) if FINAL_TRAIN_DOCS_LIMIT is None else np.arange(min(Ndv, 512))
Y_final_dev = Ydv[final_dev_idx]
planned_epochs = FINAL_MAX_EPOCHS
print("final recipe:", json.dumps(rc_norm(FINAL_RC), default=str))
print(f"planned epochs = {planned_epochs} (early stopping patience {FINAL_PATIENCE}) | train docs {len(final_train_idx):,} | dev docs {len(final_dev_idx):,}")

seed_everything(SEED)
final_model = build_model(FINAL_RC)
n_total = sum(p.numel() for p in final_model.parameters()); n_train = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
n_head = sum(p.numel() for p in final_model.head.parameters())
assert n_train - n_head == lorma_param_count(cfg, FINAL_RC["targets"], FINAL_RC["rank"])
print(f"parameters: total {n_total:,} | trainable {n_train:,} ({n_train/n_total:.2%}) = LoRMA {n_train-n_head:,} + head {n_head:,}")
FT = build_training(final_model, FINAL_RC, len(final_train_idx), planned_epochs)
final_train_ds = LSIDataset("train", final_train_idx, FINAL_RC["views"], append_end=FINAL_RC["end_token"]); final_dev_ds = LSIDataset("dev", final_dev_idx, FINAL_RC["views"], append_end=FINAL_RC["end_token"])

history, best_f1, best_epoch, bad, start_epoch = [], -1.0, 0, 0, 1
if RESUME_FINAL and FINAL_LAST_PT.exists():
    ck = torch.load(FINAL_LAST_PT, map_location="cpu", weights_only=False)
    r = final_model.load_state_dict(ck["trainable_state"], strict=False); assert not r.unexpected_keys and all(not is_trainable_name(k) for k in r.missing_keys)
    FT["opt"].load_state_dict(ck["opt"]); FT["sched"].load_state_dict(ck["sched"]); FT["scaler"].load_state_dict(ck["scaler"])
    history, best_f1, best_epoch, bad, start_epoch = ck["history"], ck["best_f1"], ck["best_epoch"], ck["bad"], ck["epoch"] + 1
    del ck; print(f"resumed after epoch {start_epoch-1}; best so far {best_f1:.4f} (epoch {best_epoch})")

run_start = time.time()
for epoch in range(start_epoch, planned_epochs + 1):
    ep0 = time.time()
    tr_loss, d = train_one_epoch(final_model, make_loader(final_train_ds, FT["mb"], True, SEED + epoch), FT, tag=f"final ep{epoch}/{planned_epochs}", log_frac=0.1)
    z_dev, _ = predict_logits(final_model, final_dev_ds, FT["mb"] * 2)
    m = evaluate_logits(Y_final_dev, z_dev)
    rec = dict(epoch=epoch, train_loss=tr_loss, **{k: m[k] for k in ["macro_f1_global", "best_global_thr", "macro_ap", "macro_f1_perlabel_cv", "micro_f1", "subset_acc", "macro_precision", "macro_recall", "bce"]},
               train_docs_per_s=d, epoch_minutes=(time.time() - ep0) / 60); history.append(rec)
    print(f"\n=== epoch {epoch}: train loss {tr_loss:.4f} | dev macro-F1 {m['macro_f1_global']:.4f} @thr {m['best_global_thr']:.2f} | per-label(CV) {m['macro_f1_perlabel_cv']:.4f} | "
          f"macro-AP {m['macro_ap']:.4f} | micro-F1 {m['micro_f1']:.4f} | {rec['epoch_minutes']:.1f} min ===\n")
    if m["macro_f1_global"] > best_f1 + 1e-4:
        best_f1, best_epoch, bad = m["macro_f1_global"], epoch, 0
        torch.save({"trainable_state": trainable_state(final_model), "epoch": epoch, "dev_macro_f1_global": best_f1}, FINAL_BEST_PT)
        np.save(FINAL_DEV_LOGITS, z_dev.astype(np.float32)); print(f"   new best epoch {epoch} saved -> {FINAL_BEST_PT}")
    else:
        bad += 1; print(f"   no improvement ({bad}/{FINAL_PATIENCE})")
    json.dump(history, open(FINAL_HISTORY, "w"), indent=1)
    torch.save({"trainable_state": trainable_state(final_model), "opt": FT["opt"].state_dict(), "sched": FT["sched"].state_dict(), "scaler": FT["scaler"].state_dict(),
                "history": history, "best_f1": best_f1, "best_epoch": best_epoch, "bad": bad, "epoch": epoch}, FINAL_LAST_PT)
    if bad >= FINAL_PATIENCE: print("early stopping."); break
print(f"\nfinal training finished: best epoch {best_epoch}, dev macro-F1 {best_f1:.4f}, wall time {(time.time()-run_start)/3600:.2f} h (this session)")
display(pd.DataFrame(history).round(4))

final recipe: {"views": ["head", "cue_rest"], "view_agg": "tokens", "head": "attn_last", "loss": "bce_posw", "head_lr": 0.001, "dropout": 0.1, "warmup": 0.06, "wd": 0.01, "end_token": true, "rank": 8, "alpha": 16, "targets": ["q", "k", "v", "o", "gate", "up", "down"], "lorma_lr": 0.001, "a_std": null}
planned epochs = 6 (early stopping patience 2) | train docs 42,750 | dev docs 10,181
parameters: total 141,212,004 | trainable 2,123,364 (1.50%) = LoRMA 1,769,472 + head 353,892
   final ep1/6 267/2672 | loss 0.4627 | 14.6 docs/s
   final ep1/6 534/2672 | loss 0.4071 | 14.7 docs/s
   final ep1/6 801/2672 | loss 0.3846 | 14.7 docs/s
   final ep1/6 1068/2672 | loss 0.3737 | 14.6 docs/s
   final ep1/6 1335/2672 | loss 0.3680 | 14.6 docs/s
   final ep1/6 1602/2672 | loss 0.3622 | 14.6 docs/s
   final ep1/6 1869/2672 | loss 0.3564 | 14.6 docs/s
   final ep1/6 2136/2672 | loss 0.3510 | 14.6 docs/s
   final ep1/6 2403/2672 | loss 0.3466 | 14.6 docs/s
   final ep1/6 2670/2672 | loss 0.3426 | 14.6

,epoch,train_loss,macro_f1_global,best_global_thr,macro_ap,macro_f1_perlabel_cv,micro_f1,subset_acc,macro_precision,macro_recall,bce,train_docs_per_s,epoch_minutes
0,1,0.3426,0.2911,0.45,0.2730,0.3152,0.3909,0.0140,0.2703,0.3945,0.1533,14.6432,53.3760
1,2,0.2898,0.3185,0.49,0.2972,0.3346,0.4094,0.0251,0.2856,0.4164,0.1535,14.6376,53.3832
2,3,0.2676,0.3420,0.49,0.3213,0.3592,0.4381,0.0290,0.3230,0.4096,0.1374,14.6352,53.3897
3,4,0.2414,0.3613,0.56,0.3363,0.3727,0.4577,0.0402,0.3457,0.4115,0.1389,14.6349,53.3953
4,5,0.2035,0.3716,0.57,0.3458,0.3815,0.4637,0.0407,0.3434,0.4260,0.1362,14.6352,53.3876
5,6,0.1519,0.3764,0.54,0.3403,0.3779,0.4694,0.0441,0.3533,0.4124,0.1271,14.6341,53.4025


## 13. Reload the best epoch, choose the post-processing on dev (out-of-fold), predict the test set, write the submission

In [20]:
ck = torch.load(FINAL_BEST_PT, map_location="cpu", weights_only=False)
_r = final_model.load_state_dict(ck["trainable_state"], strict=False); assert not _r.unexpected_keys and all(not is_trainable_name(k) for k in _r.missing_keys)
final_model.eval(); best_epoch = ck["epoch"]; best_state = ck["trainable_state"]; del ck
z_dev_final = np.load(FINAL_DEV_LOGITS); assert z_dev_final.shape == Y_final_dev.shape

def strategy_table(y, z, K=5):
    rows, preds = [], {}
    for name, fn in STRATEGIES.items():
        t0 = time.time(); pr = crossfit(y, z, fn, K=K); preds[name] = pr
        rows.append(dict(strategy=name, **prf_summary(y, pr), seconds=time.time() - t0))
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True), preds

POST_TABLE, POST_PREDS = strategy_table(Y_final_dev, z_dev_final, K=5)
print("post-processing options, out-of-fold on the full dev set:"); display(POST_TABLE.round(4))
POST_NAME = POST_TABLE.iloc[0]["strategy"]
post_predictor = STRATEGIES[POST_NAME](Y_final_dev, z_dev_final)         # refit on ALL dev documents
dev_oof_pred = POST_PREDS[POST_NAME]
print(f"\nchosen post-processing: {POST_NAME} | out-of-fold dev macro-F1 {POST_TABLE.iloc[0]['macro_f1']:.4f} | micro-F1 {POST_TABLE.iloc[0]['micro_f1']:.4f} | subset acc {POST_TABLE.iloc[0]['subset_acc']:.4f}")
base_row = POST_TABLE[POST_TABLE.strategy == "global"].iloc[0]; print(f"(plain global threshold would give macro-F1 {base_row['macro_f1']:.4f})")

# ---------------- test inference ----------------
test_ds = LSIDataset("test", np.arange(Nte), FINAL_RC["views"], with_labels=False, append_end=FINAL_RC["end_token"])
t0 = time.time(); z_test, test_dps = predict_logits(final_model, test_ds, FT["mb"] * 2); infer_s = time.time() - t0
np.save(RUN_DIR / "test_logits.npy", z_test.astype(np.float32)); print(f"test inference: {Nte:,} docs in {infer_s/60:.1f} min ({test_dps:.1f} docs/s, unmerged LoRMA)")
test_pred = post_predictor(z_test)

# ---------------- submission ----------------
def write_submission(case_ids, pred_bin, path):
    out = []
    for i, cid in enumerate(case_ids):
        labs = sorted({int(j) for j in np.flatnonzero(pred_bin[i])})
        out.append({"id": cid, "pred": labs})
    assert len(out) == len(case_ids) == len({json.dumps(c) for c in case_ids}), "one unique object per test document"
    for o in out:
        assert all(isinstance(x, int) and 0 <= x < NUM_LABELS for x in o["pred"]) and len(o["pred"]) == len(set(o["pred"]))
    json.dump(out, open(path, "w")); return out

test_ids = ARR["test"]["ids"]
submission = write_submission(test_ids, test_pred, RUN_DIR / "submission.json")
if ON_KAGGLE: write_submission(test_ids, test_pred, WORK / "submission_gqa_lorma.json")
n_lab = np.array([len(o["pred"]) for o in submission])
print(f"submission written: {len(submission):,} objects | labels/doc mean {n_lab.mean():.2f}, min {n_lab.min()}, max {n_lab.max()} | empty {int((n_lab==0).sum())}")
print("first 3:", submission[:3]); print("train mean labels/doc for reference: %.2f" % Ytr.sum(1).mean())

post-processing options, out-of-fold on the full dev set:


,strategy,macro_f1,micro_f1,macro_precision,macro_recall,subset_acc,avg_pred,empty_rate,seconds
0,per-label shrunk(300) + top1,0.3786,0.4763,0.3555,0.4145,0.0433,4.7979,0.0000,2.0917
1,per-label shrunk(100) + top1,0.3780,0.4756,0.3597,0.4093,0.0432,4.8092,0.0000,2.0439
2,global,0.3760,0.4691,0.3520,0.4130,0.0438,4.4807,0.0433,1.0415
3,global + top1,0.3756,0.4684,0.3499,0.4148,0.0451,4.5240,0.0000,1.0322
4,temperature -> global,0.3756,0.4684,0.3499,0.4148,0.0451,4.5240,0.0000,3.3726
5,per-label raw + top1,0.3753,0.4718,0.3703,0.4000,0.0418,4.8315,0.0000,2.0716
6,stacker -> per-label shrunk(100),0.3739,0.4837,0.3706,0.3904,0.0446,4.7763,0.0000,34.5961
7,Platt -> per-label shrunk(100),0.3726,0.4828,0.3769,0.3852,0.0459,4.6439,0.0000,4.7361
8,stacker -> global,0.3633,0.4840,0.3494,0.4033,0.0325,5.5885,0.0000,33.7497
9,Platt -> global,0.3590,0.4764,0.3505,0.4078,0.0297,5.9911,0.0000,3.7492



chosen post-processing: per-label shrunk(300) + top1 | out-of-fold dev macro-F1 0.3786 | micro-F1 0.4763 | subset acc 0.0433
(plain global threshold would give macro-F1 0.3760)
test inference: 13,019 docs in 6.0 min (36.3 docs/s, unmerged LoRMA)
submission written: 13,019 objects | labels/doc mean 4.74, min 1, max 28 | empty 0
first 3: [{'id': '100002997', 'pred': [5, 34, 35, 37, 94]}, {'id': '100004477', 'pred': [5, 81, 94]}, {'id': '1000086', 'pred': [34]}]
train mean labels/doc for reference: 3.70


## 14. Save the final `.pt` files, verify they reload exactly, and verify that LoRMA merges into the backbone

* `gqa_lorma_trainable.pt` — **the PEFT artifact**: only the LoRMA factors + the head (a few MB), plus the recipe and post-processing. Needs the Phase 1 `best_model.pt` to run.
* `gqa_lorma_merged_full.pt` — **self-contained**: `W' = (I + (α/r)BA)W₀` merged into every adapted layer (plain `nn.Linear`s again → no adapter overhead at inference) + head.
* Reload check (fresh model from `best_model.pt` + the saved trainable state must reproduce the in-memory logits) and merge check (merged == unmerged in fp32), plus the latency effect of merging.

In [21]:
FINAL_PT = RUN_DIR / "gqa_lorma_trainable.pt"
post = dict(name=POST_NAME, thresholds=np.asarray(post_predictor.thr).tolist(), calibrator=post_predictor.tf.params, ensure_at_least_one_label=True)
payload = {"trainable_state": best_state, "recipe": rc_norm(FINAL_RC),
           "lorma": dict(rank=FINAL_RC["rank"], alpha=FINAL_RC["alpha"], targets=list(FINAL_RC["targets"]), a_std=FINAL_RC["a_std"], variant=LORMA_VARIANT),
           "decoder_config": asdict(cfg), "num_labels": NUM_LABELS, "label_ids": list(range(NUM_LABELS)), "best_epoch": best_epoch,
           "dev_metrics_best_epoch": history[best_epoch - 1] if history else None, "postprocess": post,
           "dev_oof_macro_f1_after_postprocess": float(POST_TABLE.iloc[0]["macro_f1"]), "prefix_token_id": PREFIX_ID, "max_seq_len": MAX_LEN,
           "content_tokens_per_view": CONTENT, "tokenizer_path": str(PHASE1_TOKENIZER_PATH), "phase1_checkpoint": str(PHASE1_CHECKPOINT_PATH),
           "experiment": "phase1 GQA decoder (frozen) + LoRMA+ (scratch) + attention-pooling multi-label head"}
torch.save(payload, FINAL_PT); print(f"saved {FINAL_PT} ({FINAL_PT.stat().st_size/1024**2:.1f} MB)")

# ---- reload check: fresh backbone from the Phase 1 checkpoint + saved trainable state == in-memory model
chk_idx = np.arange(min(64, len(final_dev_idx))); chk_ds = LSIDataset("dev", final_dev_idx[chk_idx], FINAL_RC["views"], append_end=FINAL_RC["end_token"])
z_mem, _ = predict_logits(final_model, chk_ds, 8, amp=False)
saved = torch.load(FINAL_PT, map_location="cpu", weights_only=False)
rc_loaded = dict(saved["recipe"]); rc_loaded["views"] = tuple(rc_loaded["views"]); rc_loaded["targets"] = tuple(rc_loaded["targets"])
reloaded = build_model(rc_loaded, grad_ckpt=False)
_r = reloaded.load_state_dict(saved["trainable_state"], strict=False); assert not _r.unexpected_keys and all(not is_trainable_name(k) for k in _r.missing_keys)
z_re, _ = predict_logits(reloaded, chk_ds, 8, amp=False)
print(f"reload check (fp32, {len(chk_idx)} dev docs): max |logit difference| = {np.abs(z_mem - z_re).max():.2e}"); assert np.abs(z_mem - z_re).max() < 1e-3
del reloaded; torch.cuda.empty_cache()

# ---- merge check: merged == unmerged (fp32) and latency before/after merging
merged_model = merge_lorma_(copy.deepcopy(final_model)); merged_model.eval()
assert not any(isinstance(m_, LoRMALinear) for m_ in merged_model.modules())
z_mg, _ = predict_logits(merged_model, chk_ds, 8, amp=False)
print(f"merge check (fp32): max |merged - unmerged| logits = {np.abs(z_mem - z_mg).max():.2e}"); assert np.abs(z_mem - z_mg).max() < 1e-3
lat_ds = LSIDataset("dev", final_dev_idx[:min(512, len(final_dev_idx))], FINAL_RC["views"], append_end=FINAL_RC["end_token"])
_ = predict_logits(final_model, lat_ds, FT["mb"] * 2)                               # warm-up
_, dps_unmerged = predict_logits(final_model, lat_ds, FT["mb"] * 2); _, dps_merged = predict_logits(merged_model, lat_ds, FT["mb"] * 2)
print(f"inference throughput (fp16, {len(lat_ds)} dev docs): unmerged LoRMA {dps_unmerged:.1f} docs/s | merged {dps_merged:.1f} docs/s ({dps_merged/dps_unmerged-1:+.1%})")

if SAVE_MERGED_FULL:
    MERGED_PT = RUN_DIR / "gqa_lorma_merged_full.pt"
    torch.save({"model_state_dict": {k: v.detach().cpu() for k, v in merged_model.state_dict().items()}, "recipe": rc_norm(FINAL_RC), "decoder_config": asdict(cfg),
                "num_labels": NUM_LABELS, "postprocess": post, "prefix_token_id": PREFIX_ID, "note": "LoRMA merged into the backbone: W' = W0 + (alpha/r) B (A W0); load into LegalDecoder + head, no LoRMA modules needed"}, MERGED_PT)
    print(f"saved {MERGED_PT} ({MERGED_PT.stat().st_size/1024**2:.0f} MB)")
del merged_model; gc.collect(); torch.cuda.empty_cache()

saved /kaggle/working/gqa_lorma_runs/gqa_lorma_trainable.pt (8.2 MB)
reload check (fp32, 64 dev docs): max |logit difference| = 0.00e+00
merge check (fp32): max |merged - unmerged| logits = 2.24e-05
inference throughput (fp16, 512 dev docs): unmerged LoRMA 35.7 docs/s | merged 45.9 docs/s (+28.5%)
saved /kaggle/working/gqa_lorma_runs/gqa_lorma_merged_full.pt (532 MB)


## 15. Analysis for the report (§10.5): statutes missed / confused, rare statutes, cost, comparison with Experiment 1

In [22]:
y, pr = Y_final_dev, dev_oof_pred                   # out-of-fold predictions after the chosen post-processing
f1 = label_f1(y, pr); sup_tr = Ytr.sum(0); sup_dv = y.sum(0)
yb, pb = y.astype(bool), pr.astype(bool); tp = (yb & pb).sum(0); fp = (~yb & pb).sum(0); fn = (yb & ~pb).sum(0)
per_label = pd.DataFrame({"statute": np.arange(NUM_LABELS), "train_support": sup_tr, "dev_support": sup_dv, "precision": tp / np.maximum(tp + fp, 1),
                          "recall": tp / np.maximum(tp + fn, 1), "f1": f1, "FP": fp, "FN": fn})
per_label.to_csv(RUN_DIR / "final_per_label_metrics.csv", index=False)
per_label["support_quartile"] = pd.qcut(per_label["train_support"].rank(method="first"), 4, labels=["rarest 25%", "rare-mid", "common-mid", "most common 25%"])
print("macro-F1 by statute frequency (how rare statutes affect performance):"); display(per_label.groupby("support_quartile", observed=True)[["f1", "precision", "recall"]].mean().round(3))
print("worst statutes:"); display(per_label.sort_values("f1").head(12).round(3).reset_index(drop=True))
print("best statutes:");  display(per_label.sort_values("f1", ascending=False).head(8).round(3).reset_index(drop=True))
print("correlation between log statute frequency and F1: %.3f" % np.corrcoef(np.log(sup_tr), f1)[0, 1])

co = (Ytr.T.astype(np.float64) @ Ytr.astype(np.float64)); marg = np.diag(co).copy(); np.fill_diagonal(co, 0)
lift = co * len(Ytr) / np.maximum(np.outer(marg, marg), 1)
M = (yb & ~pb).T.astype(np.int64) @ (~yb & pb).astype(np.int64); np.fill_diagonal(M, 0)
top = np.dstack(np.unravel_index(np.argsort(-M.ravel())[:12], M.shape))[0]
print("\nmost common 'missed j while spurious k predicted' pairs:")
for j, k in top: print(f"  missed {j:>2d} / predicted-instead {k:>2d}: {M[j,k]:4d} docs   (train co-occurrence lift of the pair: {lift[j,k]:.1f})")
print(f"confusion pairs vs. train co-occurrence correlation: {np.corrcoef(np.log1p(M).ravel(), np.log1p(co).ravel())[0, 1]:.3f}")

# ---- cost / efficiency table (this run) + reference numbers of Experiment 1 (encoder + full fine-tuning, from its notebook output)
EXP1_REF = dict(dev_macro_f1_global=0.383, dev_oof_macro_f1_after_postprocess=0.386, trainable_params_M=139.4, train_docs_per_s=13.5,
                epoch_minutes=57, test_infer_docs_per_s=42.4, checkpoint_MB=532)
n_total = sum(p.numel() for p in final_model.parameters()); n_train = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
cost = pd.DataFrame([
    {"experiment": "Exp.1 encoder + full fine-tuning", "dev macro-F1 (global thr)": EXP1_REF["dev_macro_f1_global"], "dev macro-F1 after post-proc (OOF)": EXP1_REF["dev_oof_macro_f1_after_postprocess"],
     "trainable params (M)": EXP1_REF["trainable_params_M"], "train docs/s": EXP1_REF["train_docs_per_s"], "min/epoch": EXP1_REF["epoch_minutes"], "test docs/s": EXP1_REF["test_infer_docs_per_s"], "checkpoint MB": EXP1_REF["checkpoint_MB"]},
    {"experiment": "Exp.2 GQA decoder + LoRMA (frozen backbone)", "dev macro-F1 (global thr)": best_f1, "dev macro-F1 after post-proc (OOF)": float(POST_TABLE.iloc[0]["macro_f1"]),
     "trainable params (M)": n_train / 1e6, "train docs/s": float(np.mean([h["train_docs_per_s"] for h in history])), "min/epoch": float(np.mean([h["epoch_minutes"] for h in history])),
     "test docs/s": test_dps, "checkpoint MB": FINAL_PT.stat().st_size / 1024**2}])
display(cost.round(3))
print(f"\nGQA+LoRMA: {n_train:,} trainable of {n_total:,} parameters ({n_train/n_total:.2%}); LoRMA targets {list(FINAL_RC['targets'])}, r={FINAL_RC['rank']}, alpha={FINAL_RC['alpha']}; "
      f"merged-model inference {dps_merged:.1f} docs/s vs unmerged {dps_unmerged:.1f} docs/s")
if PROBE_TABLE is not None: print("\nProbe 1 - LoRMA (which matrices / LR):"); display(PROBE_TABLE.round(4))
if POOL_TABLE is not None: print("\nProbe 2 - pooling (attention vs attention + last-token):"); display(POOL_TABLE.round(4))
print("final pooling head:", FINAL_RC["head"], "| closing </s> appended:", FINAL_RC["end_token"])
print("\n================ SUMMARY FOR THE REPORT ================")
print(f"best epoch {best_epoch} of {len(history)} run | dev macro-F1 (single global threshold) {best_f1:.4f} | after post-processing '{POST_NAME}' (out-of-fold) {POST_TABLE.iloc[0]['macro_f1']:.4f}")
print(f"Exp.1 reference: {EXP1_REF['dev_macro_f1_global']:.3f} / {EXP1_REF['dev_oof_macro_f1_after_postprocess']:.3f}   ->   difference {best_f1-EXP1_REF['dev_macro_f1_global']:+.4f} / {POST_TABLE.iloc[0]['macro_f1']-EXP1_REF['dev_oof_macro_f1_after_postprocess']:+.4f}")

macro-F1 by statute frequency (how rare statutes affect performance):


,f1,precision,recall
support_quartile,,,
rarest 25%,0.267,0.254,0.293
rare-mid,0.353,0.334,0.386
common-mid,0.391,0.366,0.430
most common 25%,0.503,0.469,0.550


worst statutes:


,statute,train_support,dev_support,precision,recall,f1,FP,FN,support_quartile
0,8,349,92,0.037,0.065,0.047,155,86,rarest 25%
1,0,343,86,0.109,0.128,0.118,90,75,rarest 25%
2,52,1084,272,0.138,0.151,0.144,256,231,common-mid
3,2,367,98,0.162,0.133,0.146,67,85,rarest 25%
4,65,351,105,0.173,0.133,0.151,67,91,rarest 25%
5,40,543,152,0.130,0.204,0.159,208,121,rare-mid
6,82,604,156,0.136,0.199,0.161,197,125,rare-mid
7,32,385,99,0.133,0.212,0.163,137,78,rarest 25%
8,76,530,142,0.153,0.190,0.169,150,115,rare-mid
9,99,674,176,0.137,0.250,0.177,278,132,rare-mid


best statutes:


,statute,train_support,dev_support,precision,recall,f1,FP,FN,support_quartile
0,34,10380,2596,0.784,0.823,0.803,589,460,most common 25%
1,28,602,151,0.786,0.801,0.793,33,30,rare-mid
2,20,487,122,0.701,0.787,0.741,41,26,rare-mid
3,81,1321,346,0.708,0.763,0.734,109,82,common-mid
4,94,3835,958,0.706,0.675,0.690,270,311,most common 25%
5,29,497,125,0.706,0.576,0.634,30,53,rare-mid
6,61,3723,932,0.604,0.637,0.620,390,338,most common 25%
7,5,11730,2936,0.538,0.715,0.614,1802,837,most common 25%


... (3 outputs left out)

,variant,macro_F1@global,thr,macro_AP,macro_F1 per-label(CV),micro_F1,macro_P,macro_R,subset_acc,avg_#labels,trainable,train_min,docs/s,peak_GiB,selected
1,all7 | lr 1e-3,0.2980,0.48,0.2588,0.3015,0.4088,0.2658,0.3624,0.0257,5.1057,2043492,18.3390,14.6447,10.4336,<<<
0,all7 | lr 3e-4,0.2872,0.46,0.2510,0.2888,0.4034,0.2506,0.3585,0.0220,5.3860,2043492,18.2782,14.6938,10.4336,
2,attn4 | lr 1e-3,0.2840,0.40,0.2447,0.2872,0.3935,0.2279,0.4032,0.0103,6.6843,798308,15.6232,17.1903,8.0272,



Probe 2 - pooling (attention vs attention + last-token):


,variant,macro_F1@global,thr,macro_AP,macro_F1 per-label(CV),micro_F1,macro_P,macro_R,subset_acc,avg_#labels,trainable,train_min,docs/s,peak_GiB,selected
1,pool=attn + last-token (closing </s>),0.3065,0.47,0.2695,0.3064,0.4138,0.2673,0.3898,0.0210,5.4550,2123364,18.3514,14.6347,10.4348,<<<
0,pool=attn (Exp.1 head),0.2980,0.48,0.2588,0.3015,0.4088,0.2658,0.3624,0.0257,5.1057,2043492,18.3390,14.6447,10.4336,


final pooling head: attn_last | closing </s> appended: True

================ SUMMARY FOR THE REPORT ================
best epoch 6 of 6 run | dev macro-F1 (single global threshold) 0.3764 | after post-processing 'per-label shrunk(300) + top1' (out-of-fold) 0.3786
Exp.1 reference: 0.383 / 0.386   ->   difference -0.0066 / -0.0074
